# Rubin weather downtime: observed envelope, recorded state, and a long-baseline forecast

> **Diagnostic** supporting [ROSP-11](https://rubinobs.atlassian.net/browse/ROSP-11) (time-loss accounting), input to the time attribution proposal in [ROSP-24](https://rubinobs.atlassian.net/browse/ROSP-24); not an official metric page. Moved from [rubin_notebooks/earlyoperations/system_performance](https://github.com/bstalder/rubin_notebooks/tree/main/earlyoperations/system_performance), where the cache builders (`build_log_cache.py`, `build_weather_cache.py`, `allsky/build_nightly.py`, `allsky/build_dream_dense.py`) still live. Caches are read from `SYSTEM_PERFORMANCE_DATA_DIR` (default `/sdf/data/rubin/user/stalder/system_performance_data`); the final cell writes its results there too. Companion: [allsky_cloud_analysis.ipynb](allsky_cloud_analysis.ipynb).


**Window:** day_obs 2025-04-15 (LSSTCam first photon) through **2026-07-13** (last night before the July 2026 winter storm; later nights are excluded on purpose because the storm would skew the statistics).

> **Revision 2026-10-04.** The allskycam cloud record ([allsky/](https://github.com/bstalder/rubin_notebooks/tree/main/earlyoperations/system_performance/allsky), 2021-07 → 2026-07) is now a primary input: it enters the envelope (§2), the night attribution (§3), the flag comparison (§4) and the long-baseline climatology (§5), replacing the meteoblue nowcast as the cloud measurement wherever the allskycam has coverage. §4.1 adds a reduction-quality gate and an independent check of the allskycam against LSSTCam zero points. The 425 nights whose v1 astrometric solution inflated the index were re-reduced with v3 the same day. **2026-10-07:** §4.5 adds DREAM, the summit's photometric cloud monitor, as a second sky instrument over its one-year overlap: it agrees with the allskycam frame by frame (r = 0.88) and night by night (r = 0.87), measures the thin cloud the allskycam misses (DREAM-only hours are 94 % non-photometric by LSSTCam), confirms the allskycam's bright-moon floor, and leaves the §5 headline unchanged (0.106 vs 0.098 of dark time over the same nights). **2026-10-11:** the whole allskycam archive was re-reduced with v5 (camera-unit-aware astrometry and completeness, true-night keying, looser star-finder cuts; see [allsky/README.md](https://github.com/bstalder/rubin_notebooks/blob/main/earlyoperations/system_performance/allsky/README.md)). The recommendation of §5.2 moves from 0.17 / 0.13 to **0.14 / 0.12**: the drop is false cloud removed (February 2023 went from 0.54 to 0.05 cloudy against a nowcast of 0.00; P(index > 0.3 | photometric LSSTCam exposure) fell from 1.3 % to 0.3 %, and the astrometric gate now removes no nights). The numbers below are for v5.

**Questions**
1. In what external environmental ranges has Simonyi actually taken science data, and on nights it did not, was the cause weather or hardware/engineering?
2. Since the Scheduler began publishing an explicit `WEATHER` observatory-status flag (2026-02-25), is that recorded weather-downtime state consistent with the environmental telemetry, the narrative log, and the night reports?
3. Using the allskycam cloud record (Jul 2021 →), the full EFD environmental history (back to Dec 2019) and the CTIO cloud record the scheduler simulations already use, what nightly weather-downtime fraction (mean, median, standard deviation, seasonal table) should the scheduler simulations assume?

**Data sources** (all cached under `SYSTEM_PERFORMANCE_DATA_DIR`, written as `data/` below; rebuild with `build_log_cache.py` and `build_weather_cache.py`)

| Source | What it gives | Coverage |
|---|---|---|
| ConsDB `cdb_lsstcam.exposure` | per-exposure record, `can_see_sky`, `img_type` | 2025-04-15 → |
| Scheduler `logevent_observatoryStatus` (salIndex 1) | bitmask DAYTIME=1 OPERATIONAL=2 FAULT=4 **WEATHER=8** DOWNTIME=16 IDLE=32 | 2026-02-25 → |
| `narrativelog.message` | human time-lost entries with `time_lost_type` ∈ {fault, weather} | 2023 → |
| `nightreport.nightreport` | per-night free text (`weather`, `maintel_summary`) | 2025-06-26 → (Simonyi-only rows 2024-06 → 2025-06) |
| ESS weather tower (salIndex 301) | wind, gust, T, RH, dew point at 1-min | 2023-03 → |
| `WeatherStation.*` / `Environment.*` CSCs | legacy summit weather station, 1-min | 2019-12 → 2022-07 |
| **Allskycam** (the Canon 5D Mk IV fisheye all-sky camera on the summit, 1 frame/min; reduced in [allsky/](https://github.com/bstalder/rubin_notebooks/tree/main/earlyoperations/system_performance/allsky); called "allskycam" throughout to keep it distinct from LSSTCam) | per-frame **cloud index** = 1 − detected/expected bright stars under a clear-sky completeness model (0 clear … 1 overcast); `data/allsky_cloud_frames.parquet` from `allsky/build_nightly.py` | 2021-07-10 → 2026-07-18 (1666 nights; restarted 2026-10-10) |
| `cdb_lsstcam.visit1_quicklook` zero points | per-exposure extinction, the independent transparency check of the allskycam (§4.1) | 2025-04-15 → |
| **DREAM** `cloud_sys` maps (LFA `rubinobs-lfa-cp/DREAM/`) | NSIDE-32 extinction maps every ≈ 32 s; whole-sky median extinction, extinction at every LSSTCam pointing, cell-by-cell comparison with the allskycam (§4.5); `data/dream_*.parquet` from `allsky/build_dream_dense.py` | 2025-06-25 → 2026-06-21 (315 nights) |
| `WeatherForecast.hourlyTrend` (meteoblue) | hourly total/low/mid/high cloud cover; we keep the shortest-lead forecast for each valid hour as a *cloud nowcast*; retained for comparison and for nights without allskycam coverage | 2022-12 → (gap 2025-01-16 → 2025-04-16) |
| `MTDome.apertureShutter`, `ATDome.position` | dome open state | 2024-06 → / 2019-12 → |
| Watcher `logevent_alarm` | the operational humidity / dew-point thresholds actually configured | 2025 → |
| `rubin_sim_data/site_models/clouds_ctio_1975_2022.db` | CTIO/Blanco cloud oktas, the scheduler sims' default cloud model | 1975 → 2022 |
| `rubin_scheduler` Almanac | −12° twilight times → dark hours per night | |

**Conventions.** "Night" = sun altitude below −12° (the scheduler's observing window). All environmental quantities are 1-minute server-side aggregates from the EFD. `day_obs` is the Rubin convention (UTC−12 h date).

In [ ]:
import os, pathlib, re, warnings, json
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)

# Caches built by the rubin_notebooks system_performance scripts (see the header).
DATA = pathlib.Path(
    os.environ.get(
        "SYSTEM_PERFORMANCE_DATA_DIR",
        "/sdf/data/rubin/user/stalder/system_performance_data",
    )
)
DAY0, DAY1 = 20250415, 20260713  # analysis window (inclusive)
FLAG_START = 20260225  # first night with Scheduler observatoryStatus
# Operational limits used throughout (see §2.3 for where each comes from)
LIM = dict(rh=80.0, dpd=2.0, wind=15.0, gust=18.0)
CLOUD_T = 70  # % total cloud cover (nowcast) treated as "closed for clouds"; calibrated in §4.2
CAM_RMS_MAX = 1.4  # allskycam nights whose running astrometric solution has median rms > this (px) are dropped: a mis-registered solution leaves a flat 0.3–0.6 index floor on clear skies (§4.1; affected 425 v1 nights before the v3 re-reduction, 10 nights after)
CAM_T = 0.3  # allskycam cloud index treated as "cloudy" (0 = clear, 1 = overcast); calibrated against the flag in §4.2, 0.5 reported as the stricter alternative

# --- palette (dataviz reference instance; slots 1-3 validated all-pairs) ---
P = dict(
    blue="#2a78d6",
    orange="#eb6834",
    aqua="#1baf7a",
    yellow="#eda100",
    magenta="#e87ba4",
    green="#008300",
    violet="#4a3aa7",
    red="#e34948",
)
INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"
mpl.rcParams.update(
    {
        "figure.facecolor": SURF,
        "axes.facecolor": SURF,
        "axes.edgecolor": GRID,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 1,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.labelcolor": INK2,
        "xtick.color": INK2,
        "ytick.color": INK2,
        "text.color": INK,
        "axes.titlecolor": INK,
        "axes.titlelocation": "left",
        "axes.titleweight": "semibold",
        "font.size": 10,
        "lines.linewidth": 2,
        "figure.dpi": 110,
    }
)


def ym(day_obs):
    return (
        pd.to_datetime(pd.Series(day_obs).astype(str), format="%Y%m%d")
        .dt.to_period("M")
        .values
    )

## 1. The on-sky record (ConsDB)

Per night: dark hours (−12° twilight to twilight), number of science exposures with `can_see_sky`, science open-shutter hours, the on-sky span (first to last on-sky exposure, clipped to the dark window), and MTDome shutter-open hours from the EFD.

In [ ]:
alm = pd.read_parquet(DATA / "almanac_nights_2019_2027.parquet").set_index("day_obs")
ex = pd.read_parquet(DATA / "consdb_lsstcam_exposures_20250415_20260720.parquet")
ex = ex[(ex.day_obs >= DAY0) & (ex.day_obs <= DAY1)]
sh = pd.read_parquet(DATA / "env_mtdome_shutter_1min.parquet")
sh.index = pd.to_datetime(sh.index, utc=True)

nights = alm.loc[DAY0:DAY1, ["dark_hours_12", "t_set", "t_rise"]].rename(
    columns={"dark_hours_12": "dark_h"}
)
sci = ex[(ex.img_type == "science") & (ex.can_see_sky == True)]
onsky = ex[ex.can_see_sky == True]
nights["n_sci"] = sci.groupby("day_obs").size()
nights["sci_exp_h"] = sci.groupby("day_obs").exp_time.sum() / 3600
nights["n_onsky"] = onsky.groupby("day_obs").size()
nights["n_any"] = ex.groupby("day_obs").size()
first = onsky.groupby("day_obs").obs_start.min()
last = onsky.groupby("day_obs").obs_end.max()
nights["span_h"] = (
    (last.clip(upper=nights.t_rise) - first.clip(lower=nights.t_set)).dt.total_seconds()
    / 3600
).clip(lower=0)
nights = nights.fillna(
    {"n_sci": 0, "sci_exp_h": 0, "n_onsky": 0, "n_any": 0, "span_h": 0}
)
nights["shutter_open_h"] = [
    (sh.loc[r.t_set : r.t_rise, "shutter0"] > 95).sum() / 60
    for r in nights.itertuples()
]
nights["observed"] = nights.n_sci > 0
print(
    f"{len(nights)} nights in window; {int(nights.observed.sum())} with science exposures ({nights.observed.mean():.0%}); "
    f"{int((nights.n_any>0).sum())} with any exposure (incl. calibrations); {nights.dark_h.sum():.0f} dark hours total"
)
monthly = nights.groupby(ym(nights.index)).agg(
    nights=("dark_h", "size"),
    sci_nights=("observed", "sum"),
    dark_h=("dark_h", "sum"),
    sci_exp_h=("sci_exp_h", "sum"),
    span_h=("span_h", "sum"),
    shutter_open_h=("shutter_open_h", "sum"),
)
monthly.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.6))
x = np.arange(len(monthly))
w = 0.26
ax.bar(x - w, monthly.dark_h, w, color=GRID, label="dark hours (−12°)", zorder=2)
ax.bar(
    x, monthly.shutter_open_h, w, color=P["blue"], label="MTDome shutter open", zorder=2
)
ax.bar(
    x + w,
    monthly.sci_exp_h,
    w,
    color=P["orange"],
    label="science open-shutter",
    zorder=2,
)
ax.set_xticks(x)
ax.set_xticklabels([str(p) for p in monthly.index], rotation=45, ha="right")
ax.set_ylabel("hours per month")
ax.set_title(
    "Dark time, dome-open time and science exposure time, LSSTCam 2025-04 → 2026-07-13"
)
ax.legend(frameon=False, ncol=3, loc="upper left")
plt.tight_layout()

## 2. Environmental envelope while taking science data

Every on-sky science exposure is matched to the 1-minute weather-tower bin containing its mid-exposure time, to the allskycam frame nearest mid-exposure (within 90 s), to the meteoblue cloud nowcast valid for that hour, and to the DIMM minute.
The ConsDB `wind_speed`, `humidity`, `air_temp` columns agree with the EFD (r = 0.98 / 0.999 / 1.00), but the ConsDB wind is held/stale in the high-wind tail, so the EFD values are used.

In [ ]:
wt = pd.read_parquet(DATA / "env_weather_tower_1min.parquet")
wt.index = pd.to_datetime(wt.index, utc=True)
wt["dpd"] = wt.air_temp - wt.dew_point
fc = pd.read_parquet(DATA / "forecast_nowcast_hourly.parquet")
fc.index = pd.to_datetime(fc.index, utc=True)
dm = pd.read_parquet(DATA / "env_dimm_1min.parquet")
dm.index = pd.to_datetime(dm.index, utc=True)
cam = pd.read_parquet(
    DATA / "allsky_cloud_frames.parquet"
)  # allskycam: one row per reduced frame (sun < −12°, 1/min), written by allsky/build_nightly.py
cam["t"] = pd.to_datetime(cam.mjd + 2400000.5, unit="D", origin="julian", utc=True)
cam = cam.set_index("t").sort_index()
night_rms = (
    cam[cam.ast_mode == "refine"].groupby("day_obs").ast_rms_px.median()
)  # quality of the night's running astrometric solution (NaN = no solvable frame = overcast, kept)
cam_bad_nights = night_rms.index[night_rms > CAM_RMS_MAX]
cam_all, cam = cam, cam[~cam.day_obs.isin(cam_bad_nights)]


def cam_at(times, tol="90s", c=None):
    return (cam if c is None else c).cloud_idx.reindex(
        pd.DatetimeIndex(times), method="nearest", tolerance=pd.Timedelta(tol)
    )  # nearest frame within tol, else NaN


print(
    f"allskycam: {len(cam_all):,} dark frames on {cam_all.day_obs.nunique()} nights, {cam_all.day_obs.min()} → {cam_all.day_obs.max()}; "
    f"astrometric gate (night median rms > {CAM_RMS_MAX} px) removes {len(cam_bad_nights)} nights = {1 - len(cam)/len(cam_all):.0%} of frames (see §4.1), {len(cam):,} frames kept"
)
sci = sci.copy()
mid = pd.DatetimeIndex(
    (sci.obs_start + (sci.obs_end - sci.obs_start) / 2).dt.floor("1min")
).tz_convert("UTC")
m = wt.reindex(mid)
for c in ["wind_speed", "wind_gust", "air_temp", "rh", "dew_point", "dpd", "wind_dir"]:
    sci[c + "_efd"] = m[c].values
sci["tcc_fc"] = fc.tcc.reindex(mid, method="ffill").values
sci["dimm_efd"] = dm.dimm_fwhm.reindex(mid).values
sci["cam_idx"] = cam_at(
    (sci.obs_start + (sci.obs_end - sci.obs_start) / 2).dt.tz_convert("UTC")
).values
print(
    f"allskycam frame within 90 s of mid-exposure for {sci.cam_idx.notna().mean():.1%} of science exposures"
)
assert sci.wind_speed_efd.notna().mean() > 0.999, "EFD match failed"
print(
    "ConsDB vs EFD correlation  wind %.3f  RH %.3f  T %.3f"
    % (
        sci[["wind_speed", "wind_speed_efd"]].corr().iloc[0, 1],
        sci[["humidity", "rh_efd"]].corr().iloc[0, 1],
        sci[["air_temp", "air_temp_efd"]].corr().iloc[0, 1],
    )
)
cols = {
    "wind_speed_efd": "wind speed (m/s, 1-min mean)",
    "wind_gust_efd": "wind gust (m/s, 1-min max)",
    "air_temp_efd": "air temperature (°C)",
    "rh_efd": "relative humidity (%)",
    "dew_point_efd": "dew point (°C)",
    "dpd_efd": "dew-point depression T−Td (°C)",
    "tcc_fc": "total cloud cover nowcast (%)",
    "cam_idx": "allskycam cloud index (0 clear … 1 overcast)",
    "dimm_efd": "DIMM seeing (arcsec)",
}
env = (
    sci[list(cols)]
    .describe(percentiles=[0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999])
    .T.rename(index=cols)
)
env["N"] = env["count"].astype(int)
env = env.drop(columns=["count", "mean", "std"]).round(2)
env

### 2.1 The observed operating envelope

The table below is the empirical envelope. The 99.9th / 0.1th percentiles are the practical "we have taken data at" limits; the max/min are single excursions. For the two cloud measures the 95th percentile is quoted instead: the survey does observe through broken cloud, and the allskycam index is 0 for the median exposure but above 0.3 for about 5 % of them (see §4.1 for what that costs in extinction).

In [ ]:
pn = sci.groupby("day_obs").agg(
    wind_max=("wind_speed_efd", "max"),
    gust_max=("wind_gust_efd", "max"),
    rh_max=("rh_efd", "max"),
    dpd_min=("dpd_efd", "min"),
    t_min=("air_temp_efd", "min"),
    t_max=("air_temp_efd", "max"),
    tcc_med=("tcc_fc", "median"),
    cam_med=("cam_idx", "median"),
    n=("exposure_id", "size"),
)
envelope = pd.DataFrame(
    {
        "quantity": [
            "sustained wind (m/s)",
            "gust (m/s)",
            "relative humidity (%)",
            "dew-point depression (°C)",
            "air temperature (°C)",
            "cloud cover nowcast (%)",
            "cloud index, allskycam",
        ],
        "per-exposure 99.9% (or 0.1%) bound": [
            sci.wind_speed_efd.quantile(0.999),
            sci.wind_gust_efd.quantile(0.999),
            sci.rh_efd.quantile(0.999),
            sci.dpd_efd.quantile(0.001),
            f"{sci.air_temp_efd.quantile(.001):.1f} … {sci.air_temp_efd.quantile(.999):.1f}",
            sci.tcc_fc.quantile(0.95),
            f"{sci.cam_idx.quantile(.95):.2f} (95th pct)",
        ],
        "absolute extreme while exposing": [
            sci.wind_speed_efd.max(),
            sci.wind_gust_efd.max(),
            sci.rh_efd.max(),
            sci.dpd_efd.min(),
            f"{sci.air_temp_efd.min():.1f} … {sci.air_temp_efd.max():.1f}",
            sci.tcc_fc.max(),
            f"{sci.cam_idx.max():.2f}",
        ],
        "nights whose max/min exceeded": [
            (pn.wind_max > LIM["wind"]).sum(),
            (pn.gust_max > LIM["gust"]).sum(),
            (pn.rh_max > 70).sum(),
            (pn.dpd_min < 5).sum(),
            (pn.t_min < 2).sum(),
            (pn.tcc_med > CLOUD_T).sum(),
            (pn.cam_med > CAM_T).sum(),
        ],
        "threshold used in previous column": [
            f">{LIM['wind']}",
            f">{LIM['gust']}",
            ">70",
            "<5",
            "<2",
            f"median >{CLOUD_T}",
            f"median >{CAM_T}",
        ],
    }
).set_index("quantity")
envelope = envelope.apply(
    lambda col: col.map(
        lambda v: f"{v:.1f}" if isinstance(v, (float, np.floating)) else v
    )
)
envelope

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13.5, 5.6))
spec = [
    ("wind_speed_efd", "wind speed (m/s)", LIM["wind"], 0, 22),
    ("wind_gust_efd", "gust (m/s)", LIM["gust"], 0, 26),
    ("rh_efd", "relative humidity (%)", LIM["rh"], 0, 100),
    ("dpd_efd", "dew-point depression (°C)", LIM["dpd"], 0, 56),
    ("air_temp_efd", "air temperature (°C)", None, -4, 20),
    ("dimm_efd", "DIMM seeing (arcsec)", None, 0.3, 2.5),
    ("tcc_fc", "cloud cover nowcast (%)", CLOUD_T, 0, 100),
    ("cam_idx", "allskycam cloud index", CAM_T, 0, 1),
]
for ax, (c, lab, lim, lo, hi) in zip(axes.ravel(), spec):
    ax.hist(sci[c].dropna(), bins=np.linspace(lo, hi, 45), color=P["blue"], zorder=2)
    if lim is not None:
        ax.axvline(lim, color=P["red"], lw=1.5, zorder=3)
        ax.text(
            lim,
            ax.get_ylim()[1] * 0.95,
            f" limit {lim:g}",
            color=INK2,
            fontsize=8,
            va="top",
        )
    ax.set_xlabel(lab)
    ax.set_yticks([])
    ax.grid(False)
axes[0, 0].set_title(
    f"Conditions during {len(sci):,} on-sky science exposures (weather tower 1-min, nowcast, allskycam; red = limit / cloudy threshold)"
)
plt.tight_layout()

### 2.2 Operational limits: where they come from

The project does not publish one table of go/no-go limits, so the limits used here are the ones the system itself enforces or the operators state in the logs:

| quantity | limit | evidence |
|---|---|---|
| relative humidity | **80 %** (weather tower) | Watcher alarm `Humidity.WeatherStation` warning at >80 % (212k firings); M1M3/M2/camera inlet humidity alarms at 70/80/90 %; operators close at "humidity reached 80 %" (2025-11-01, 2026-02-04, 2026-03-20) |
| dew-point depression | **2.0 °C close** (2.5 °C serious, 3.0 °C check) | Watcher alarm `DewPointDepression.Simonyi External`: "Close the dome: dew point … < 2.00" |
| sustained wind | **~15 m/s** | narrative log: "wind has been at the borderline limit of 15 m/s … mount motion image degradation" (2026-04-04), AuxTel shutdowns at >15 m/s; Simonyi "gusts over 18 m/s, close dome" (2026-07-01), "wind spiking to 20 m/s, closing the dome" (2025-04-30) |
| gust | **~18 m/s** | same entries |
| clouds | no instrument; operators close when the scheduler/AOS cannot find stars | meteoblue nowcast total cloud cover is the only continuous cloud record; threshold calibrated in §4.2 |
| temperature / precipitation / snow | camera-thermal and safety driven, not a numeric limit | "below the lower limit to open the dome shutter" at 5.6 °C in July 2025 (camera trim-heat); snow/ice on dome → closed |

The Watcher alarm thresholds are recovered directly from the alarm `reason` text:

In [ ]:
wa = pd.read_parquet(DATA / "watcher_alarms_20250101_20260720.parquet")
wa = wa[~wa.reason.str.contains("No ", na=False)]
rows = []
for nm in [
    "Humidity.WeatherStation",
    "DewPointDepression.Simonyi External",
    "Humidity.M1M3 Humidity",
    "Humidity.M2 Humidity",
    "Humidity.Camera humidity",
]:
    s = wa[wa.alarm_name == nm]
    th = s.reason.str.extract(r"([<>]) ?([\d.]+)")
    for (sev, op, t), n in (
        s.assign(op=th[0], thr=th[1]).groupby(["severity", "op", "thr"]).size().items()
    ):
        if n > 20:
            rows.append(
                dict(
                    alarm=nm,
                    severity={
                        1: "warning(1)",
                        2: "warning(2)",
                        3: "serious(3)",
                        4: "critical(4)",
                    }.get(sev, sev),
                    threshold=f"{op} {t}",
                    n_events=n,
                )
            )
pd.DataFrame(rows).set_index(["alarm", "severity"])

## 3. Nights without science data: weather or hardware?

A night is "observed" if it has at least one on-sky science exposure. Nights with on-sky exposures of other types only (`acq`, `cwfs`, `engtest`: AOS, pointing and LUT work) are "on-sky engineering": the sky was usable but no science was taken. For the remaining closed nights the cause is assigned from four independent records, in priority order:

1. **Scheduler `WEATHER` flag** (2026-02-25 onward): >50 % of dark minutes flagged WEATHER.
2. **Allskycam**: >50 % of dark minutes with cloud index > 0.3 (needs ≥80 % frame coverage of the night and a night that passes the astrometric gate of §4.1, which after v5 removes none).
3. **EFD environmental proxy** (nights without allskycam coverage): >50 % of dark minutes outside the limits of §2.2 *or* with cloud nowcast > 70 % (needs ≥80 % telemetry coverage).
4. **Log text**: narrative-log `time_lost_type='weather'` ≥ 4 h attributable to Simonyi, or a night-report sentence of the form "could not open / closed due to <cloud|humidity|snow|…>".
5. **Hardware / engineering**: night report says camera/cryo/TMA/vacuum/dynalene/shutdown/off-sky/closed-dome tests, or narrative-log fault time ≥ 4 h, or Scheduler `DOWNTIME` > 4 h.
6. If none of the above but the allskycam (or, without allskycam, the EFD proxy) says the sky was usable (<25 % cloudy), the night is **engineering/commissioning with a usable sky** (this covers most 2025 commissioning nights, whose reports describe daytime work without a "closed" sentence).

The table also shows the attribution the notebook made before the allskycam was reduced (`nights_without_allskycam`), and the cross-tabulation of the nights that moved. The allskycam mostly *confirms* nowcast-proxy weather nights and *resolves* nights that were undetermined or looked like hardware closures but were in fact cloudy.

In [ ]:
# ---- records -----------------------------------------------------------------
nr = pd.read_csv(DATA / "nightreport_raw.csv")
nr["fmt"] = np.where(nr.telescope.isna(), "new", "old")
nr = (
    nr.sort_values(["day_obs", "fmt", "date_added"])
    .drop_duplicates("day_obs", keep="last")
    .set_index("day_obs")
)
nl = pd.read_csv(DATA / "narrativelog_timelost.csv")
for c in ["date_begin", "date_end"]:
    nl[c] = pd.to_datetime(nl[c], format="mixed", utc=True)
nl["day_obs"] = (
    (nl.date_begin - pd.Timedelta(hours=12)).dt.strftime("%Y%m%d").astype(int)
)


def which_tel(t):
    t = str(t).lower()
    at = bool(
        re.search(
            r"auxtel|aux tel|\bat\b|latiss|atdome|atmcs|atptg|ataos|athexapod|atpneumatics|atcamera|atspectrograph",
            t,
        )
    )
    mt = bool(
        re.search(
            r"simonyi|\btma\b|lsstcam|mtaos|m1m3|\bm2\b|mtdome|mtmount|mtptg|mthexapod|mtrotator|comcam|mtm1m3|mtm2|scheduler:1|maintel|mtcamera|\bfbs\b|main telescope|the dome|dynalene",
            t,
        )
    )
    return "both" if at and mt else "AT" if at else "MT" if mt else "unk"


nl["tel"] = nl.message_text.map(which_tel)
simonyi = nl.tel.isin(
    ["MT", "both", "unk"]
)  # 'unk' entries are kept: generic "weather loss" entries are usually observatory-wide
nl_w = (
    nl[(nl.time_lost_type == "weather") & (nl.time_lost_hours > 0) & simonyi]
    .drop_duplicates(["day_obs", "time_lost_hours"])
    .groupby("day_obs")
    .time_lost_hours.sum()
)
nl_f = (
    nl[(nl.time_lost_type == "fault") & (nl.time_lost_hours > 0) & simonyi]
    .drop_duplicates(["day_obs", "time_lost_hours"])
    .groupby("day_obs")
    .time_lost_hours.sum()
)
WXC = (
    r"(?i)(not|unable|could ?n.t|did ?n.t|never|no) (safe to )?(open|go on[- ]?sky|observ)[^.]{0,80}(cloud|humid|dew|snow|ice|wind|rain|weather|storm|fog|condens|cold|freez)"
    r"|(cloud|humid|dew|snow|ice|wind|rain|weather|storm|fog|condens|cold|freez)[^.]{0,80}(not|unable|could ?n.t|did ?n.t|never|no) (safe to )?(open|go on[- ]?sky|observ)"
    r"|clos(ed|e|ing)[^.]{0,40}(due to|because of)[^.]{0,40}(cloud|humid|dew|snow|ice|wind|rain|weather|storm|fog|condens)"
    r"|(cloud|humid|dew|snow|ice|wind|rain|weather|storm|fog|condens)[^.]{0,60}(prevent|kept|keep|remain|stay)[^.]{0,20}clos|weather (time )?loss|lost[^.]{0,20}weather|loss[^.]{0,20}weather|no on-sky[^.]{0,40}weather|poor weather|bad weather|evacuat"
)
HWC = (
    r"(?i)(no|not) (on-sky|observ|simonyi)[^.]{0,80}(camera|cryo|lsstcam|shutdown|vacuum|dynalene|tma|oss|pump|screw|firmware)"
    r"|(camera|cryo|lsstcam|vacuum|dynalene|tma|oss|pump cart|screw|firmware)[^.]{0,80}(no|not|lost) (on-sky|observ|the night|tonight)"
    r"|closed[- ]dome (day|night|test)|off-sky|shutdown|telescope not used|observations canceled|no simonyi|lost the (whole )?night|cancel(l)?ed"
)


def report_flags(d):
    if d not in nr.index:
        return pd.Series(
            dict(has_report=False, wx_closed=False, hw_closed=False, mt_text="")
        )
    r = nr.loc[d]
    mt = re.sub(
        r"\s+",
        " ",
        (
            str(r.maintel_summary)
            if pd.notna(r.maintel_summary)
            else (str(r.summary) if pd.notna(r.summary) else "")
        ),
    )
    wx = re.sub(r"\s+", " ", str(r.weather) if pd.notna(r.weather) else "")
    return pd.Series(
        dict(
            has_report=True,
            wx_closed=bool(re.search(WXC, mt + " " + wx)),
            hw_closed=bool(re.search(HWC, mt)),
            mt_text=mt[:140],
        )
    )


nights = nights.join(
    pd.DataFrame([report_flags(d) for d in nights.index], index=nights.index)
)
nights["nl_weather_h"] = nl_w.reindex(nights.index).fillna(0)
nights["nl_fault_h"] = nl_f.reindex(nights.index).fillna(0)
print(
    f"night reports found for {int(nights.has_report.sum())}/{len(nights)} nights; narrative-log Simonyi weather entries on {(nights.nl_weather_h>0).sum()} nights totalling {nights.nl_weather_h.sum():.0f} h; fault entries {nights.nl_fault_h.sum():.0f} h"
)

In [ ]:
# ---- Scheduler observatory status → per-minute state on a night grid -------------------------
st = pd.read_parquet(DATA / "scheduler_observatory_status.parquet")
st = st[st.salIndex == 1].sort_index()


def night_grid(d0, d1, with_status=True):
    a = alm.loc[d0:d1]
    grid = pd.date_range(
        a.t_set.min().floor("1min"), a.t_rise.max().ceil("1min"), freq="1min", tz="UTC"
    )
    night = np.zeros(len(grid), bool)
    dob = np.zeros(len(grid), int)
    for r in a.itertuples():
        i, j = grid.searchsorted(r.t_set), grid.searchsorted(r.t_rise)
        night[i:j] = True
        dob[i:j] = r.Index
    g = pd.DataFrame({"day_obs": dob[night]}, index=grid[night])
    if with_status:
        s = st[["status"]].reindex(g.index, method="ffill").status
        g["status"] = s
        g = g[g.status.notna()]
        g["status"] = g.status.astype(int)
        g["W"] = (g.status & 8) > 0
        g["F"] = (g.status & 4) > 0
        g["D"] = (g.status & 16) > 0
        g["U"] = g.status == 0
    return g


g26 = night_grid(FLAG_START, DAY1)
g26 = g26.join(wt[["wind_speed", "wind_gust", "rh", "dpd"]]).join(sh[["shutter0"]])
g26["tcc"] = fc.tcc.reindex(g26.index, method="ffill")
g26["lim"] = (
    (g26.rh > LIM["rh"])
    | (g26.dpd < LIM["dpd"])
    | (g26.wind_speed > LIM["wind"])
    | (g26.wind_gust > LIM["gust"])
)
g26["cloud"] = g26.tcc > CLOUD_T
g26["open"] = g26.shutter0 > 95
g26["cam"] = cam_at(g26.index).values
g26["cam_cloud"] = g26.cam > CAM_T
sched = (
    g26.groupby("day_obs")[
        ["W", "F", "D", "U", "lim", "cloud", "cam_cloud", "open"]
    ].sum()
    / 60
).rename(columns=lambda c: c + "_h")
sched["night_h"] = g26.groupby("day_obs").size() / 60
nights = nights.join(sched)
print(
    f"Scheduler flag covers {len(sched)} nights ({sched.night_h.sum():.0f} dark h); WEATHER {sched.W_h.sum():.0f} h, FAULT {sched.F_h.sum():.0f} h, DOWNTIME {sched.D_h.sum():.0f} h, UNKNOWN(CSC not monitoring) {sched.U_h.sum():.0f} h"
)

In [ ]:
# ---- EFD environmental proxy for every night (full baseline, used again in §5) ---------------------------
u = pd.read_parquet(
    DATA / "env_unified_1min_2019_2026.parquet"
)  # Environment (2019-12→2021-02) + WeatherStation (2021-02→2022-07) + ESS 301 (2023-03→)
# Data quality: the Environment-era (Dec 2019–Mar 2020) wind has hours-long values >25 m/s in midsummer with ordinary gusts → unreliable; drop wind for that era
bad_wind = u.src == "Environment"
u.loc[bad_wind, ["wind_speed", "wind_gust"]] = np.nan
n = u[u.night].copy()
n["tcc"] = fc.tcc.reindex(n.index, method="ffill")
age_h = (
    n.index - fc.index.to_series().reindex(n.index, method="ffill")
).dt.total_seconds() / 3600
n.loc[~(age_h <= 3), "tcc"] = np.nan  # no forecast issued within 3 h → cloud unknown
n["lim_rh"] = n.rh > LIM["rh"]
n["lim_dpd"] = n.dpd < LIM["dpd"]
n["lim_wind"] = (n.wind_speed > LIM["wind"]) | (n.wind_gust > LIM["gust"])
n["lim"] = n.lim_rh | n.lim_dpd | n.lim_wind
n["cloud"] = n.tcc > CLOUD_T
n["cam"] = cam_at(n.index).values
n["cam_cloud"] = np.where(n.cam.notna(), n.cam > CAM_T, np.nan)
n["bad_cam"] = np.where(
    n.cam.notna() & n.rh.notna(), n.lim | (n.cam > CAM_T), np.nan
)  # allskycam cloud ∪ RH/dew/wind limits
n["bad"] = np.where(n.tcc.notna() & n.rh.notna(), n.lim | n.cloud, np.nan)
proxy = n.groupby("day_obs").agg(
    nmin=("rh", "size"),
    lim_cov=("rh", lambda s: s.notna().mean()),
    fc_cov=("tcc", lambda s: s.notna().mean()),
    lim_frac=("lim", "mean"),
    rh_frac=("lim_rh", "mean"),
    dpd_frac=("lim_dpd", "mean"),
    wind_frac=("lim_wind", "mean"),
    cloud_frac=("cloud", lambda s: s[n.loc[s.index, "tcc"].notna()].mean()),
    bad_frac=("bad", "mean"),
    cam_cov_lim=("cam", lambda s: s.notna().mean()),
    bad_cam_frac=("bad_cam", "mean"),
    src=("src", "first"),
)
proxy["dark_h"] = alm.dark_hours_12.reindex(proxy.index)
proxy["cover"] = proxy.nmin / 60 / proxy.dark_h
proxy["valid_full"] = (proxy.cover > 0.8) & (proxy.lim_cov > 0.8) & (proxy.fc_cov > 0.8)
proxy["valid_lim"] = (proxy.cover > 0.8) & (proxy.lim_cov > 0.8)
proxy["valid_cam_full"] = proxy.valid_lim & (
    proxy.cam_cov_lim > 0.8
)  # limits and allskycam both cover the night
# ---- allskycam, every night it ran (independent of the EFD series, which has a gap 2022-07 → 2023-03) ----
camn = cam.groupby("day_obs").agg(
    n_dark=("cloud_idx", "size"),
    cam_frac=("cloud_idx", lambda s: (s > CAM_T).mean()),
    cam_frac50=("cloud_idx", lambda s: (s > 0.5).mean()),
    cam_idx_mean=("cloud_idx", "mean"),
)
camn["dark_h"] = alm.dark_hours_12.reindex(camn.index)
camn["cam_cov"] = camn.n_dark / 60 / camn.dark_h
camn["cam_valid"] = camn.cam_cov > 0.8
camn["month"] = camn.index.astype(str).str[4:6].astype(int)
camn["year"] = camn.index.astype(str).str[:4].astype(int)
nights = nights.join(
    proxy[
        [
            "bad_frac",
            "lim_frac",
            "cloud_frac",
            "valid_full",
            "bad_cam_frac",
            "valid_cam_full",
        ]
    ].rename(columns={"valid_full": "proxy_valid"})
)
nights = nights.join(camn[["cam_frac", "cam_frac50", "cam_valid", "cam_cov"]])
nights["cam_valid"] = nights.cam_valid.fillna(False).astype(bool)
nights["valid_cam_full"] = nights.valid_cam_full.fillna(False).astype(bool)
print(
    f"proxy nights: {len(proxy)} total, {int(proxy.valid_full.sum())} with full (limits+cloud) coverage, {int(proxy.valid_lim.sum())} with limits coverage; "
    f"allskycam: {len(camn)} nights, {int(camn.cam_valid.sum())} with ≥80 % frame coverage, {int(proxy.valid_cam_full.sum())} with allskycam+limits; in the LSSTCam window {int(nights.cam_valid.sum())}/{len(nights)} nights have allskycam coverage"
)

In [ ]:
def classify(r, use_cam=True):
    if r.observed:
        return "observed"
    if r.n_onsky > 0:
        return "on-sky engineering only (no science)"
    if pd.notna(r.W_h) and r.W_h / r.night_h > 0.5:
        return "closed: weather (Scheduler flag)"
    if use_cam and r.cam_valid and r.cam_frac > 0.5:
        return "closed: weather (allskycam)"
    if r.proxy_valid and r.bad_frac > 0.5:
        return "closed: weather (EFD proxy)"
    if r.nl_weather_h >= 4 or r.wx_closed:
        return "closed: weather (log text)"
    if r.hw_closed or r.nl_fault_h >= 4 or (pd.notna(r.D_h) and r.D_h > 4):
        return "closed: hardware/engineering"
    sky_ok = (
        (r.cam_frac < 0.25)
        if (use_cam and r.cam_valid)
        else (bool(r.proxy_valid) and r.bad_frac < 0.25)
    )  # allskycam is the sky-usability measurement; nowcast proxy only where the allskycam is missing
    if sky_ok:
        return "closed: engineering/commissioning (sky usable)"
    return "closed: undetermined"


nights["class"] = nights.apply(classify, axis=1)
nights["class_nowcast"] = nights.apply(
    lambda r: classify(r, use_cam=False), axis=1
)  # the attribution the notebook made before the allskycam was reduced
order = [
    "observed",
    "on-sky engineering only (no science)",
    "closed: weather (Scheduler flag)",
    "closed: weather (allskycam)",
    "closed: weather (EFD proxy)",
    "closed: weather (log text)",
    "closed: hardware/engineering",
    "closed: engineering/commissioning (sky usable)",
    "closed: undetermined",
]
summary = (
    nights.groupby("class")
    .agg(nights=("dark_h", "size"), dark_h=("dark_h", "sum"))
    .reindex(order)
    .fillna(0)
)
summary["share_of_nights"] = (summary.nights / len(nights)).round(3)
summary["nights_without_allskycam"] = (
    nights.groupby("class_nowcast").size().reindex(order).fillna(0).astype(int)
)
chg = nights[nights["class"] != nights.class_nowcast]
print(
    f"{len(chg)} of {len(nights)} nights change class when the allskycam is used; weather-closed nights {int(nights.class_nowcast.str.startswith('closed: weather').sum())} → {int(nights['class'].str.startswith('closed: weather').sum())}"
)
print(
    pd.crosstab(
        chg.class_nowcast.str.replace("closed: ", ""),
        chg["class"].str.replace("closed: ", ""),
        rownames=["without allskycam"],
        colnames=["with allskycam"],
    ),
    "\n",
)
summary.astype({"nights": int, "dark_h": int})

In [ ]:
wx_nights = nights["class"].str.startswith("closed: weather")
print(
    f"Nights fully lost: weather {int(wx_nights.sum())} ({wx_nights.sum()/len(nights):.1%} of {len(nights)} nights, {nights.dark_h[wx_nights].sum():.0f} dark h); "
    f"hardware/engineering/commissioning {int(nights['class'].isin(order[6:8]).sum())}; undetermined {int((nights['class']==order[-1]).sum())}"
)
ct = pd.crosstab(ym(nights.index), nights["class"]).reindex(columns=order, fill_value=0)
fig, ax = plt.subplots(figsize=(10, 3.8))
colors = {
    "observed": P["blue"],
    "on-sky engineering only (no science)": "#86b6ef",
    "closed: weather (Scheduler flag)": P["red"],
    "closed: weather (allskycam)": "#ec6e6d",
    "closed: weather (EFD proxy)": "#f29c9b",
    "closed: weather (log text)": "#f8cbca",
    "closed: hardware/engineering": P["yellow"],
    "closed: engineering/commissioning (sky usable)": "#f5d98a",
    "closed: undetermined": "#c3c2b7",
}
bottom = np.zeros(len(ct))
for c in order:
    ax.bar(
        np.arange(len(ct)),
        ct[c],
        0.7,
        bottom=bottom,
        color=colors[c],
        label=c,
        zorder=2,
        edgecolor=SURF,
        linewidth=1,
    )
    bottom += ct[c].values
ax.set_xticks(np.arange(len(ct)))
ax.set_xticklabels([str(p) for p in ct.index], rotation=45, ha="right")
ax.set_ylabel("nights")
ax.set_title("Why each night did or did not produce science data")
ax.legend(
    frameon=False, fontsize=8, ncol=2, loc="upper left", bbox_to_anchor=(0, -0.35)
)
plt.tight_layout()

In [ ]:
# The undetermined nights, for the record: no Scheduler flag, no allskycam coverage or an intermediate allskycam cloud fraction (25–50 %), and no decisive log sentence
nights[nights["class"] == "closed: undetermined"][
    ["dark_h", "cam_frac", "cam_cov", "bad_frac", "nl_fault_h", "mt_text"]
].round(2)

### 3.1 Partial nights

On observed nights the dome was typically open for most of the dark time (median on-sky span ≈ 90 % of dark hours). Weather loss *within* observed nights is now measured directly by the allskycam: cloudy minutes (index > 0.3), with the humidity / dew-point / wind limits added. The nowcast proxy, which could only see thick cloud, found about three quarters as much (172 h vs 235 h).

In [ ]:
obs = nights[nights.observed & nights.cam_valid]
cam_h = (obs.cam_frac * obs.dark_h).sum()
obs_u = nights[nights.observed & nights.valid_cam_full]
union_h = (obs_u.bad_cam_frac * obs_u.dark_h).sum()
obs_n = nights[nights.observed & nights.proxy_valid.fillna(False).astype(bool)]
now_h = (obs_n.bad_frac * obs_n.dark_h).sum()
print(
    f"observed nights with allskycam coverage: {len(obs)}; allskycam-cloudy (index > {CAM_T}) hours inside observed nights: {cam_h:.0f} h ({cam_h/obs.dark_h.sum():.1%} of their dark time); "
    f"allskycam ∪ RH/dew/wind limits: {union_h:.0f} h ({union_h/obs_u.dark_h.sum():.1%}, {len(obs_u)} nights); the nowcast proxy gave {now_h:.0f} h ({now_h/obs_n.dark_h.sum():.1%}, {len(obs_n)} nights)"
)
print(
    f"median on-sky span / dark on observed nights = {(nights[nights.observed].span_h/nights[nights.observed].dark_h).median():.2f}; "
    f"on-sky span/dark vs (1 − allskycam cloud fraction): r = {np.corrcoef(obs.span_h/obs.dark_h, 1-obs.cam_frac)[0,1]:.2f}"
)
full_lost_h = nights.dark_h[wx_nights].sum()
part_frac = union_h / obs_u.dark_h.sum()
part_h = part_frac * nights.dark_h[nights.observed].sum()
print(
    f"=> weather loss in window ≈ {full_lost_h:.0f} h (full nights) + {part_h:.0f} h (partial: allskycam ∪ limits, scaled to all observed nights) "
    f"= {(full_lost_h + part_h)/nights.dark_h.sum():.1%} of all dark time; {full_lost_h/nights.dark_h.sum():.1%} from fully lost nights alone"
)
print(
    f"   direct allskycam accounting over the same window (dark time with index > {CAM_T}, all {int(nights.cam_valid.sum())} allskycam nights): "
    f"{(nights.cam_frac*nights.dark_h).sum()/nights.dark_h[nights.cam_valid].sum():.1%}; with index > 0.5: {(nights.cam_frac50*nights.dark_h).sum()/nights.dark_h[nights.cam_valid].sum():.1%}"
)

## 4. Is the recorded weather-downtime state consistent?

The Scheduler CSC has published `logevent_observatoryStatus` since **2026-02-25**. Operators set the `WEATHER` bit by hand (notes such as "clouds rolled in", "humidity 84 % and overcast"); `FAULT` is set automatically from CSC states; `DOWNTIME` is set by hand for planned engineering. We compare the `WEATHER` bit minute-by-minute with (a) the environmental limits, (b) the allskycam cloud index, (c) the cloud nowcast, (d) the dome shutter, and nightly with (e) the narrative log and (f) the night reports.

In [ ]:
b = g26[
    ~g26.D & ~g26.U
]  # exclude planned downtime and periods when the CSC was not monitoring
print(
    f"{len(b)/60:.0f} dark hours with Scheduler status (excluding DOWNTIME/UNKNOWN): P(WEATHER) = {b.W.mean():.3f}; P(outside EFD limits) = {b.lim.mean():.3f}; "
    f"P(cloud nowcast > {CLOUD_T}%) = {b.cloud.mean():.3f}; P(allskycam index > {CAM_T}) = {b.cam_cloud[b.cam.notna()].mean():.3f} (allskycam covers {b.cam.notna().mean():.1%} of these minutes)"
)
print(
    f"P(outside limits | WEATHER) = {b.lim[b.W].mean():.2f}   P(WEATHER | outside limits) = {b.W[b.lim].mean():.2f}"
)
bc = b[b.cam.notna()]
print(
    f"P(allskycam cloudy | WEATHER) = {bc.cam_cloud[bc.W].mean():.2f}   P(WEATHER | allskycam cloudy) = {bc.W[bc.cam_cloud].mean():.2f}   P(allskycam cloudy | OPERATIONAL) = {bc.cam_cloud[bc.status==2].mean():.3f}   "
    f"P(allskycam cloudy | WEATHER and inside limits) = {bc.cam_cloud[bc.W & ~bc.lim].mean():.2f}"
)
print(
    f"P(nowcast cloudy | WEATHER) = {b.cloud[b.W].mean():.2f}   P(WEATHER | nowcast cloudy) = {b.W[b.cloud].mean():.2f}   P(nowcast cloudy | OPERATIONAL) = {b.cloud[b.status==2].mean():.3f}"
)
print(
    f"P(shutter closed | WEATHER) = {(~b.open[b.W]).mean():.2f}   P(WEATHER | shutter open) = {b.W[b.open].mean():.3f}   P(shutter open | OPERATIONAL) = {b.open[b.status==2].mean():.2f}   "
    f"P(allskycam cloudy | shutter open) = {bc.cam_cloud[bc.open].mean():.3f}"
)
print(
    "allskycam cloud index while WEATHER but inside limits — quartiles:",
    bc.cam[bc.W & ~bc.lim].quantile([0.25, 0.5, 0.75]).round(2).tolist(),
    "  while OPERATIONAL:",
    bc.cam[bc.status == 2].quantile([0.25, 0.5, 0.75]).round(2).tolist(),
)
print(
    "cloud-cover nowcast  while WEATHER but inside limits — quartiles:",
    b.tcc[b.W & ~b.lim].quantile([0.25, 0.5, 0.75]).round(0).tolist(),
    "  while OPERATIONAL:",
    b.tcc[b.status == 2].quantile([0.25, 0.5, 0.75]).round(0).tolist(),
)

**Reading:** only ~7 % of WEATHER minutes are explained by the humidity / dew-point / wind limits; the WEATHER state is overwhelmingly a **cloud** state. The allskycam confirms it directly: while flagged and inside the limits the median cloud index is 0.71 (quartiles 0.18–0.96) against 0.00 while OPERATIONAL, and 70 % of WEATHER minutes are allskycam-cloudy versus 7 % of OPERATIONAL minutes. The reverse direction is weaker: only half of the allskycam-cloudy minutes are flagged (the rest fall in FAULT/DOWNTIME, in the latency before the flag is set, or are broken cloud the survey observed through — 8 % of shutter-open minutes are allskycam-cloudy). Conversely ~60 % of out-of-limit minutes are flagged, and 94 % of WEATHER minutes have the shutter closed, so the flag is a faithful record of *not observing because of sky conditions*. The nowcast agrees with the flag when it says cloudy (precision 0.67) but misses more of the flagged time than the allskycam does (recall 0.58 vs 0.70).

### 4.1 Is the allskycam's cloud index itself trustworthy?

Two checks before the allskycam is used as a reference. **(a) Astrometric gate.** The v1 reduction inherited a plate scale and tilt that the per-frame refinement could not always correct. On 425 such nights the running solution sat at 1.6–1.9 px rms instead of 0.6–0.9, roughly half the predicted stars fell outside the 3-px match radius, and the cloud index showed a flat 0.3–0.6 floor all night with the north/west half of the sky "cloudy" while the night report and LSSTCam said clear (e.g. 2025-07-17, 2025-07-02; un-gated five-year fraction 0.23 instead of 0.17). Those nights were re-reduced with v3 (which frees the plate scale and distortion in the final refinement) on 2026-10-04, and the whole archive again with v5 on 2026-10-11; 2025-07-17 and 2025-07-02 now have no frame above index 0.3. The gate stays in as a safeguard: the table below shows the index still rising with rms on the few frames above 1.4 px (311 of 862,000 solved frames), and nights whose median rms exceeds `CAM_RMS_MAX` are removed from the allskycam record (nights with *no* solvable frame are overcast and are kept). After v5 it removes no nights (10 after v3).

**(b) LSSTCam zero points.** Every on-sky science exposure with a `visit1_quicklook` zero point gives an extinction relative to the band's clear-sky zero point (90th percentile per band). This is an independent, same-sky measurement of transparency, limited to the 3.5° field being observed.

In [ ]:
# (a) index vs astrometric quality, all nights before the gate
ref = cam_all[cam_all.ast_mode == "refine"].copy()
ref["rms_bin"] = pd.cut(ref.ast_rms_px, [0, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0])
qa = ref.groupby("rms_bin", observed=True).agg(
    frames=("cloud_idx", "size"),
    idx_median=("cloud_idx", "median"),
    frac_idx_gt_T=("cloud_idx", lambda s: (s > CAM_T).mean()),
)
print("solved frames by rms of the astrometric solution (px):")
print(qa.round(3).T.to_string(), "\n")
nq = (
    cam_all.groupby("day_obs")
    .agg(
        q10=("cloud_idx", lambda s: s.quantile(0.1)),
        q25=("cloud_idx", lambda s: s.quantile(0.25)),
        n=("cloud_idx", "size"),
    )
    .join(night_rms.rename("rms"))
)
floor = (nq.q10 > 0.1) & (
    nq.q10 < 0.7
)  # never clear and never overcast all night: real cloud has clear gaps (q10 ≈ 0) and overcast is 1 — an intermediate floor is the artefact's signature
print(
    f"nights with an intermediate index floor (10th percentile in 0.1–0.7): {int(floor.sum())} of {len(nq)}; of these {int((floor & (nq.rms > CAM_RMS_MAX)).sum())} have rms > {CAM_RMS_MAX} px and are removed. "
    f"Gate removes {len(cam_bad_nights)} nights by year: {nq[nq.rms > CAM_RMS_MAX].groupby(nq.index[nq.rms > CAM_RMS_MAX] // 10000).size().to_dict()}; "
    f"nights with no solvable frame (kept, overcast): {int(nq.rms.isna().sum())}, median index {nq[nq.rms.isna()].q25.median():.2f}"
)
# (b) LSSTCam photometric zero points
zp = pd.read_parquet(DATA / "consdb_lsstcam_zp_20250415_20260713.parquet")
refzp = zp.groupby("band").zp.quantile(0.9)
zp["ext"] = (
    refzp.reindex(zp.band).values - zp.zp
)  # mag of extinction vs the band's clear-sky zero point
zmid = pd.DatetimeIndex(zp.obs_start + (zp.obs_end - zp.obs_start) / 2)
zp["cam_pre"] = cam_at(zmid, c=cam_all).values
zp["cam"] = cam_at(zmid).values
for lab, col in [("before gate", "cam_pre"), ("after gate ", "cam")]:
    z = zp.dropna(subset=[col])
    c = z[col] > CAM_T
    print(
        f"{lab}: {len(z):,} exposures — P(allskycam > {CAM_T} | extinction < 0.1 mag) = {c[z.ext < 0.1].mean():.3f};  P(extinction > 0.3 | allskycam > {CAM_T}) = {(z.ext[c] > 0.3).mean():.2f};  "
        f"P(allskycam > {CAM_T} | extinction > 0.3) = {c[z.ext > 0.3].mean():.2f};  P(allskycam > {CAM_T} | extinction > 1 mag) = {c[z.ext > 1].mean():.2f};  r = {np.corrcoef(z[col], z.ext)[0,1]:.2f}"
    )
z = zp.dropna(subset=["cam"]).copy()
z["cb"] = pd.cut(z.cam, [-0.01, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0])
nz = z.groupby("day_obs").agg(
    n=("ext", "size"),
    cam_frac=("cam", lambda s: (s > CAM_T).mean()),
    ext_frac=("ext", lambda s: (s > 0.3).mean()),
)
nz = nz[nz.n >= 20]
print(
    f"night level, {len(nz)} observed nights with ≥20 zero points: r(allskycam-cloudy fraction, fraction of exposures with > 0.3 mag extinction) = {np.corrcoef(nz.cam_frac, nz.ext_frac)[0,1]:.2f}; "
    f"allskycam > 50 % cloudy but photometric (< 15 % of exposures extincted): {int(((nz.cam_frac > 0.5) & (nz.ext_frac < 0.15)).sum())} nights {list(nz.index[(nz.cam_frac > 0.5) & (nz.ext_frac < 0.15)])}; "
    f"allskycam < 20 % cloudy but > 50 % of exposures extincted: {int(((nz.cam_frac < 0.2) & (nz.ext_frac > 0.5)).sum())} nights"
)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
axes[0].bar(
    np.arange(len(qa)),
    qa.frac_idx_gt_T * 100,
    color=[P["blue"] if iv.right <= CAM_RMS_MAX else P["red"] for iv in qa.index],
    zorder=2,
)
axes[0].set_xticks(np.arange(len(qa)))
axes[0].set_xticklabels([f"{iv.left:g}–{iv.right:g}" for iv in qa.index], fontsize=8)
axes[0].set_xlabel("rms of the frame's astrometric solution (px)")
axes[0].set_ylabel(f"% of solved frames with index > {CAM_T}")
axes[0].set_title("Index inflates with poor astrometry (red = gated)", fontsize=10)
tab = z.groupby("cb", observed=True).ext.quantile([0.1, 0.5, 0.9]).unstack()
x = np.arange(len(tab))
axes[1].fill_between(
    x,
    tab[0.1],
    tab[0.9],
    color=P["blue"],
    alpha=0.15,
    lw=0,
    label="10–90 % of exposures",
)
axes[1].plot(x, tab[0.5], color=P["blue"], marker="o", ms=5, label="median")
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{iv.left:g}–{iv.right:g}" for iv in tab.index], fontsize=8)
axes[1].set_xlabel("allskycam cloud index at mid-exposure")
axes[1].set_ylabel("LSSTCam extinction (mag)")
axes[1].set_title(
    f"LSSTCam extinction vs allskycam index ({len(z):,} exposures)", fontsize=10
)
axes[1].legend(frameon=False, fontsize=8.5, loc="upper left")
plt.tight_layout()

**Reading.** The gate is a reduction-quality cut, not a weather cut, and after the v5 re-reduction it removes nothing. An allskycam index above the threshold occurs on 0.3 % of photometric exposures (1.3 % in v3), and 90 % of allskycam-cloudy exposures carry more than 0.3 mag of extinction (three quarters in v3). The reverse is weaker: the allskycam flags 36 % of the exposures with more than 0.3 mag of extinction, because thin cirrus dims stars without hiding them and because a 3.5° field can be clouded while the all-sky mean is not; even above 1 mag the allskycam catches only about two thirds. The allskycam is therefore a measure of *sky unusable for star counting*, i.e. the operator's "can the AOS find stars" criterion, and a lower bound on the time with non-photometric transparency.

### 4.2 Calibrating telemetry-only proxies to the flag

Proxy = outside limits **or** cloud metric > T, for the nowcast (T in % cover) and the allskycam (T in cloud index). Scan T on the minutes where both are defined:

In [ ]:
def kappa(a, b):
    po = (a == b).mean()
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return (po - pe) / (1 - pe)


bb = b[
    b.cam.notna() & b.tcc.notna()
]  # minutes where both proxies are defined, so the two scans are on the same footing


def scan_table(pred_of_T, Ts):
    rows = []
    for T in Ts:
        pred = pred_of_T(T) | bb.lim
        rows.append(
            dict(
                T=T,
                P_proxy=pred.mean(),
                P_WEATHER=bb.W.mean(),
                recall=pred[bb.W].mean(),
                precision=bb.W[pred].mean(),
                kappa=kappa(bb.W.values, pred.values),
                proxy_h=pred.sum() / 60,
                W_h=bb.W.sum() / 60,
            )
        )
    return pd.DataFrame(rows).set_index("T").round(3)


scan = scan_table(lambda T: bb.tcc > T, [10, 20, 30, 40, 50, 60, 70, 80, 90, 99])
scan_cam = scan_table(
    lambda T: bb.cam > T, [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
)
print(
    f"{len(bb)/60:.0f} dark hours with Scheduler status, allskycam and nowcast all defined"
)
pd.concat(
    {
        "nowcast: limits ∪ cloud cover > T %": scan,
        "allskycam: limits ∪ cloud index > T": scan_cam,
    },
    names=["proxy", "T"],
)

In [ ]:
best = scan.kappa.idxmax()
best_cam = scan_cam.kappa.idxmax()
print(
    f"nowcast: κ peaks at T≈{best}%; T={CLOUD_T}% gives {scan.loc[CLOUD_T,'proxy_h']:.0f} h proxy vs {scan.loc[CLOUD_T,'W_h']:.0f} h WEATHER, precision {scan.loc[CLOUD_T,'precision']:.2f} / recall {scan.loc[CLOUD_T,'recall']:.2f}, κ = {scan.loc[CLOUD_T,'kappa']:.2f}"
)
print(
    f"allskycam:  κ peaks at T≈{best_cam}; T={CAM_T} gives {scan_cam.loc[CAM_T,'proxy_h']:.0f} h proxy vs {scan_cam.loc[CAM_T,'W_h']:.0f} h WEATHER, precision {scan_cam.loc[CAM_T,'precision']:.2f} / recall {scan_cam.loc[CAM_T,'recall']:.2f}, κ = {scan_cam.loc[CAM_T,'kappa']:.2f}; "
    f"T=0.5 gives {scan_cam.loc[0.5,'proxy_h']:.0f} h, precision {scan_cam.loc[0.5,'precision']:.2f} / recall {scan_cam.loc[0.5,'recall']:.2f}, κ = {scan_cam.loc[0.5,'kappa']:.2f}"
)
pn26 = nights.loc[FLAG_START:DAY1].dropna(subset=["W_h"])
pn26 = pn26[(pn26.D_h < pn26.night_h * 0.5)]
gb = g26[~g26.D & ~g26.U]
pred70 = (
    ((gb.assign(p=lambda d: (d.tcc > CLOUD_T) | d.lim)).groupby("day_obs").p.sum() / 60)
    .reindex(pn26.index)
    .fillna(0)
)
predcam = (
    ((gb.assign(p=lambda d: (d.cam > CAM_T) | d.lim)).groupby("day_obs").p.sum() / 60)
    .reindex(pn26.index)
    .fillna(0)
)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 5), sharey=True)
for ax, x, lab in [
    (axes[0], pred70, f"nowcast proxy hours (limits or cloud > {CLOUD_T} %)"),
    (axes[1], predcam, f"allskycam hours (limits or cloud index > {CAM_T})"),
]:
    ax.plot([0, 12], [0, 12], color=GRID, lw=1, zorder=1)
    ax.scatter(
        x, pn26.W_h, s=28, color=P["blue"], edgecolor=SURF, linewidth=1, zorder=3
    )
    for d, x_, y_ in zip(pn26.index, x, pn26.W_h):
        if abs(x_ - y_) > 5:
            ax.annotate(
                str(d),
                (x_, y_),
                textcoords="offset points",
                xytext=(5, -3),
                fontsize=7,
                color=INK2,
            )
    ax.set_xlabel(lab)
    ax.set_xlim(-0.3, 12.5)
    ax.set_ylim(-0.3, 12.5)
    ax.set_title(
        f"r = {np.corrcoef(x, pn26.W_h)[0,1]:.2f}, {x.sum():.0f} h vs {pn26.W_h.sum():.0f} h flagged"
    )
axes[0].set_ylabel("Scheduler WEATHER hours")
fig.suptitle(
    f"Per-night agreement with the WEATHER flag, {len(pn26)} nights 2026-02-25 → 07-13",
    x=0.01,
    ha="left",
    fontweight="semibold",
)
plt.tight_layout()
nights["cam_proxy_h"] = predcam
nights["nowcast_proxy_h"] = pred70

The two proxies reach the same night-level correlation with the flag (r ≈ 0.75) but for different reasons. The nowcast at 70 % reproduces the flagged *hours* almost exactly because its threshold was tuned to do so; it misses thin and broken cloud (nights below the diagonal on the left, e.g. 2026-06-03 and 2026-05-29, where it saw no cloud at all). The allskycam at index 0.3 sees about 35 % more cloudy time than was flagged (212 vs 157 h over 118 nights): the excess is nights the flag does not cover and nights the survey observed through broken cloud. On 2026-06-20 both proxies see > 10 cloudy hours with 0 h WEATHER (nobody intended to observe); on 2026-06-03 the allskycam sees 11.1 cloudy hours, 3.9 h were flagged, and LSSTCam saw > 0.3 mag of extinction on every exposure; 2026-05-29 and 06-13 are similar (5.5–5.7 h, nothing flagged). The two flagged-but-clear nights are the same on both panels (2026-06-10, 2026-06-18: snow and ice on the dome after a storm, a closure the sky cannot show). Cohen's κ peaks at index ≈ 0.5 (κ 0.57) rather than 0.3, i.e. the operators' *de facto* closing threshold is thick cloud; index > 0.5 reproduces the flagged hours to within 7 % (165 vs 154 h). Both thresholds are carried through §5 so the choice is explicit.

### 4.3 Agreement with the human records

In [ ]:
pn26 = nights.loc[FLAG_START:DAY1].dropna(subset=["W_h"]).copy()
pn26["sched_wx"] = pn26.W_h > 1
pn26["nl_wx"] = pn26.nl_weather_h > 0
pn26["nr_wx"] = pn26.wx_closed.astype(bool)
print("nights with Scheduler WEATHER > 1 h vs narrative-log weather time-lost entry:")
print(pd.crosstab(pn26.sched_wx, pn26.nl_wx, margins=True), "\n")
print(
    "nights with Scheduler WEATHER > 1 h vs night-report 'closed for weather' sentence:"
)
print(pd.crosstab(pn26.sched_wx, pn26.nr_wx, margins=True), "\n")
print(
    f"Hours: Scheduler WEATHER {pn26.W_h.sum():.0f} h;  narrative-log Simonyi weather time_lost {pn26.nl_weather_h.sum():.0f} h;  "
    f"nowcast proxy {pred70.reindex(pn26.index).fillna(0).sum():.0f} h;  allskycam (∪ limits) {predcam.reindex(pn26.index).fillna(0).sum():.0f} h;  allskycam index > 0.5 (∪ limits) "
    f"{((gb.assign(p=lambda d: (d.cam > 0.5) | d.lim)).groupby('day_obs').p.sum() / 60).reindex(pn26.index).fillna(0).sum():.0f} h  — over the same {len(pn26)} nights."
)
print(
    f"Nights with WEATHER > 6 h: {(pn26.W_h>6).sum()}, of which {( (pn26.W_h>6) & ~pn26.observed).sum()} produced no science exposures."
)

**Finding.** The records agree on *which* nights were bad but not on *how much* time was lost:

* The **Scheduler flag**, the **allskycam** and the **nowcast proxy** agree night-by-night at r ≈ 0.75. On the 118 nights with little planned downtime (§4.2) the allskycam records more cloudy time than the flag (212 vs 157 h at index 0.3; 165 h at index 0.5) because it counts cloud during FAULT, before the flag is set, and while the survey observes through broken cloud; the nowcast records less (151 h) because it misses thin cloud. (The hour totals printed above are over all 139 flag nights, but both proxies are counted only on these 118, so they should be compared with the 157 flagged hours, not 207.) The allskycam is the measurement of the sky; the flag is the record of the decision.
* The **narrative log** `time_lost_type='weather'` captures only ~20 % of the hours (observers log weather loss mainly for AuxTel, and Simonyi entries are often generic "weather loss" notes). It should not be used for weather-downtime accounting.
* The **night report** rarely states weather time-loss numbers (8 explicit mentions in 444 reports); its `weather` field is descriptive and useful only for spot checks.

Before 2026-02-25 there is no machine-readable weather state at all, so for the first ten months of LSSTCam the allskycam (validated here against the flag and in §4.1 against LSSTCam photometry) is the continuous estimate, with the nowcast proxy as a fallback on nights the allskycam does not cover.

### 4.4 Validation on 2025 (before the flag existed)

If the allskycam is right, 2025 observed nights should have low cloudy fractions and a share of the no-science nights should have high ones (the rest being hardware closures). This is a weak test, because partial cloud on an observed night shortens the on-sky span only mildly (the correlation below is ≈ 0.25), so the quantitative calibration rests on the 2026 flag comparison in §4.2 and the LSSTCam zero points in §4.1. The listed observed-but-cloudy nights are real: those with zero points have > 0.3 mag of LSSTCam extinction on half or more of their exposures (49–100 %); 2025-08-09 was overcast until 06 UTC and LSSTCam observed in the clear second half (the night report says "cloudy"). In v3 this list also held 2025-12-01/02/03, the full-moon floor of the completeness model (index ≈ 0.35–0.45 all night under a photometric sky); after v5 those nights have no frame above 0.3.

In [ ]:
v = nights.loc[: FLAG_START - 1]
vc = v[v.cam_valid]
vp = v[v.proxy_valid.fillna(False).astype(bool)]
print(
    f"{len(vc)} 2025 nights with allskycam coverage ({len(vp)} with a valid nowcast proxy)."
)
print(
    f"Observed nights — allskycam cloud fraction: median {vc[vc.observed].cam_frac.median():.2f}, 90th pct {vc[vc.observed].cam_frac.quantile(.9):.2f}, >50 % on {(vc[vc.observed].cam_frac>0.5).sum()} of {vc.observed.sum()} nights;  "
    f"nowcast proxy: median {vp[vp.observed].bad_frac.median():.2f}, 90th pct {vp[vp.observed].bad_frac.quantile(.9):.2f}, >50 % on {(vp[vp.observed].bad_frac>0.5).sum()} nights."
)
print(
    f"Nights without science — allskycam: median {vc[~vc.observed].cam_frac.median():.2f}, >50 % on {(vc[~vc.observed].cam_frac>0.5).sum()} of {(~vc.observed).sum()};  "
    f"nowcast proxy: >50 % on {(vp[~vp.observed].bad_frac>0.5).sum()} of {(~vp.observed).sum()} (the rest are hardware/commissioning closures, §3)."
)
s = vc[vc.observed]
sp = vp[vp.observed]
print(
    f"On observed nights, on-sky span/dark vs (1 − cloudy fraction): allskycam r = {np.corrcoef(s.span_h/s.dark_h, 1-s.cam_frac)[0,1]:.2f}, nowcast proxy r = {np.corrcoef(sp.span_h/sp.dark_h, 1-sp.bad_frac)[0,1]:.2f}"
)
both = vc[vc.proxy_valid.fillna(False).astype(bool)]
print(
    f"allskycam vs nowcast proxy, night by night (2025, {len(both)} nights): r = {np.corrcoef(both.cam_frac, both.bad_frac)[0,1]:.2f}; nights allskycam > 50 % cloudy but nowcast proxy < 25 %: {((both.cam_frac>0.5)&(both.bad_frac<0.25)).sum()}; the reverse: {((both.cam_frac<0.25)&(both.bad_frac>0.5)).sum()}"
)
hi = vc[vc.observed & (vc.cam_frac > 0.5)]
hi[
    [
        "dark_h",
        "span_h",
        "sci_exp_h",
        "n_sci",
        "cam_frac",
        "cam_frac50",
        "bad_frac",
        "lim_frac",
        "cloud_frac",
    ]
].round(2)

### 4.5 A second cloud instrument: DREAM

DREAM (the Dutch Rubin Enhanced Atmospheric Monitor) is a five-camera photometric cloud monitor at the summit. Its `cloud_sys` product is an NSIDE-32 HEALPix map of **extinction in magnitudes** (per-pixel zero-point deficit of the stars it photometers against a clear-sky reference, stacked over the last five epochs ≈ 2–3 min), published to the LFA every ≈ 32 s from **2025-06-25 to 2026-06-21** (315 nights, 349 k maps; it stopped before the storm). It measures the quantity the allskycam does not: how much the stars are *dimmed*, not whether they are *lost*. `allsky/build_dream_dense.py` pulls every map, rotates it to horizon coordinates, and writes three caches: whole-sky scalars per map (`data/dream_maps.parquet`), the DREAM extinction in a 1.75° disc around every LSSTCam science pointing (`data/dream_lsstcam_exposures.parquet`), and, for every allskycam frame, the whole-sky DREAM metrics plus a cell-by-cell comparison on the allskycam's NSIDE-8 grid (`data/dream_allskycam_cells.parquet`). Four questions: (a) do the two sky monitors see the same clouds, frame by frame and cell by cell; (b) which one agrees better with LSSTCam's own zero points; (c) how does DREAM compare with the Scheduler flag; (d) does adding DREAM change the nightly weather-loss fraction.

Where the two disagree the sign is diagnostic. The allskycam is a star-*count* metric and the DREAM scalar used here is the *median extinction above 30° altitude*, so DREAM-only cloud should be thin or uniform cloud (stars dimmed but still detected) and allskycam-only cloud should be the moon floor of §4.1 or stars lost without much extinction. Both are checked below against LSSTCam and against whether the survey was actually observing.


In [ ]:
# ---- DREAM: dense cloud_sys record (every map, ~32 s), built by allsky/build_dream_dense.py from the LFA ------------------------------------------
dm = pd.read_parquet(DATA / "dream_maps.parquet")
dm["t"] = pd.to_datetime(dm.mjd + 2400000.5, unit="D", origin="julian", utc=True)
dm = dm.set_index("t").sort_index()
de = pd.read_parquet(
    DATA / "dream_lsstcam_exposures.parquet"
)  # LSSTCam science exposures: median DREAM extinction in a 1.75° disc around the pointing, nearest map within 90 s
dc = pd.read_parquet(
    DATA / "dream_allskycam_cells.parquet"
)  # allskycam frames: nearest map within 60 s, whole-sky DREAM metrics and the NSIDE-8 cell-by-cell comparison
dc = dc[~dc.day_obs.isin(cam_bad_nights)].copy()
DREAM_T = 0.5  # DREAM "cloudy": median extinction above 30° altitude > 0.5 mag (κ-optimal against both the allskycam and the flag, see below)
dc["cam_c"] = dc.cloud_idx > CAM_T
dc["dr_c"] = dc.dream_med_ext_alt30 > DREAM_T
dc["moon_up"] = dc.moon_alt > 0
cad = (
    dm.groupby("day_obs")
    .apply(lambda d: d.index.to_series().diff().median().total_seconds())
    .median()
)
print(
    f"DREAM: {len(dm):,} cloud maps on {dm.day_obs.nunique()} nights {dm.day_obs.min()} → {dm.day_obs.max()}, median cadence {cad:.0f} s; "
    f"{len(dc):,} allskycam frames with a DREAM map within 60 s ({dc.day_obs.nunique()} nights); {len(de):,} LSSTCam science exposures with a DREAM map within 90 s"
)
# (a) frame level: what does DREAM report as a function of the allskycam index?
dc["cb"] = pd.cut(dc.cloud_idx, [-0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 1.0])
fa = dc.groupby("cb", observed=True).agg(
    frames=("cloud_idx", "size"),
    dream_med_ext=("dream_med_ext_alt30", "median"),
    q10=("dream_med_ext_alt30", lambda s: s.quantile(0.1)),
    q90=("dream_med_ext_alt30", lambda s: s.quantile(0.9)),
    dream_frac_sky_gt_0p3=("dream_frac_alt30_ext_gt_0p3", "median"),
    P_dream_cloudy=("dr_c", "mean"),
    r_cells=("r_cells", "median"),
)
fa["cell_agreement"] = dc.groupby("cb", observed=True).apply(
    lambda d: (d.n_cc + d.n_00).sum() / (d.n_cc + d.n_00 + d.n_c0 + d.n_0c).sum()
)
print(
    "\nDREAM by allskycam cloud-index bin (cell agreement: NSIDE-8 cells, allskycam index > 0.5 vs DREAM mean extinction > 0.3 mag):"
)
print(fa.round(3).T.to_string())
print(
    f"\nframe-level correlation of the allskycam index with DREAM median extinction r = {np.corrcoef(dc.cloud_idx, dc.dream_med_ext_alt30.fillna(0))[0,1]:.2f}, with the DREAM fraction of sky above 0.3 mag r = {np.corrcoef(dc.cloud_idx, dc.dream_frac_alt30_ext_gt_0p3.fillna(0))[0,1]:.2f}"
)
part = dc[(dc.cloud_idx > 0.2) & (dc.cloud_idx < 0.8)]
tot = (part.n_cc + part.n_00 + part.n_c0 + part.n_0c).sum()
print(
    f"partial-cloud frames (0.2 < index < 0.8, {len(part):,}): median cell-by-cell correlation {part.r_cells.median():.2f}; cells agree {(part.n_cc + part.n_00).sum()/tot:.0%}; "
    f"of allskycam-cloudy cells DREAM calls {part.n_c0.sum()/(part.n_cc + part.n_c0).sum():.0%} clear; of DREAM-cloudy cells the allskycam calls {part.n_0c.sum()/(part.n_0c + part.n_cc).sum():.0%} clear (0.3 mag is a lower bar than 'half the stars gone')"
)
# (b) the disagreements: moon, whether LSSTCam was observing, and what LSSTCam measured
cls = np.select(
    [dc.cam_c & dc.dr_c, dc.cam_c & ~dc.dr_c, ~dc.cam_c & dc.dr_c],
    ["both cloudy", "allskycam only", "DREAM only"],
    "both clear",
)
ex_mid = (
    pd.DatetimeIndex(sci.obs_start + (sci.obs_end - sci.obs_start) / 2)
    .tz_convert("UTC")
    .sort_values()
)
ft = pd.DatetimeIndex(
    pd.to_datetime(dc.mjd + 2400000.5, unit="D", origin="julian", utc=True)
)
pos = ex_mid.searchsorted(ft)
near = np.full(len(ft), np.inf)
for k in (pos - 1, pos):
    kk = np.clip(k, 0, len(ex_mid) - 1)
    near = np.minimum(near, np.abs((ex_mid[kk] - ft).total_seconds()))
dc["observing"] = (
    near <= 120
)  # an LSSTCam science exposure mid-point within 2 min of the allskycam frame
zx = zp.dropna(subset=["cam"]).copy()
zx["t"] = pd.to_datetime(zx.obs_start + (zx.obs_end - zx.obs_start) / 2, utc=True)
mm = pd.merge_asof(
    zx[["t", "ext"]].sort_values("t"),
    dc.assign(t=ft)[["t", "cam_c", "dr_c"]].sort_values("t"),
    on="t",
    direction="nearest",
    tolerance=pd.Timedelta("90s"),
).dropna()
cls2 = np.select(
    [mm.cam_c & mm.dr_c, mm.cam_c & ~mm.dr_c, ~mm.cam_c & mm.dr_c],
    ["both cloudy", "allskycam only", "DREAM only"],
    "both clear",
)
dis = dc.groupby(cls).agg(
    hours=("cam_c", lambda s: len(s) / 60),
    moon_up=("moon_up", "mean"),
    cam_idx_median=("cloud_idx", "median"),
    dream_ext_median=("dream_med_ext_alt30", "median"),
    P_LSSTCam_observing=("observing", "mean"),
)
dis = dis.join(
    mm.groupby(cls2).ext.agg(
        lsstcam_exposures="size",
        lsstcam_ext_median="median",
        P_lsstcam_ext_gt_0p3=lambda s: (s > 0.3).mean(),
    )
)
print(
    f"\nallskycam (index > {CAM_T}) × DREAM (median extinction > {DREAM_T} mag) classification of the allskycam minutes:"
)
print(dis.round(3).to_string())

In [ ]:
# (c) three-way check with LSSTCam zero points: DREAM extinction in the 1.75° disc around the pointing vs the exposure's own extinction
j = (
    de[de.n_pix >= 4]
    .merge(zp[["exposure_id", "ext", "cam"]], on="exposure_id")
    .dropna(subset=["cam"])
)
j = j[~j.day_obs.isin(cam_bad_nights)].copy()
j["moon_up"] = (
    cam.moon_alt.reindex(
        pd.DatetimeIndex(
            pd.to_datetime(j.dream_mjd + 2400000.5, unit="D", origin="julian", utc=True)
        ),
        method="nearest",
        tolerance=pd.Timedelta("90s"),
    ).values
    > 0
)
ph = j[j.ext < 0.1]
print(
    f"{len(j):,} LSSTCam science exposures ({j.day_obs.nunique()} nights) with DREAM, allskycam and a zero point. On photometric exposures (LSSTCam extinction < 0.1 mag) DREAM reports {ph.ext_dream.median():.3f} mag median, robust σ {1.4826*(ph.ext_dream-ph.ext_dream.median()).abs().median():.3f} mag, "
    f"P(DREAM > 0.1) = {(ph.ext_dream > 0.1).mean():.2f}, P(DREAM > 0.3) = {(ph.ext_dream > 0.3).mean():.4f};  P(allskycam > {CAM_T} | photometric) = {(ph.cam > CAM_T).mean():.3f}"
)
s = j[(j.ext > 0.3) & (j.ext < 3)]
print(
    f"DREAM vs LSSTCam extinction: r = {j[j.ext < 5][['ext_dream', 'ext']].corr().iloc[0,1]:.2f}; median DREAM / LSSTCam ratio {(s.ext_dream/s.ext).median():.2f} for 0.3 < ext < 3 mag (moon down {(s.ext_dream/s.ext)[~s.moon_up].median():.2f}, moon up {(s.ext_dream/s.ext)[s.moon_up].median():.2f})"
)
rows = []
for lo, hi in [
    (-9, 0.05),
    (0.05, 0.1),
    (0.1, 0.2),
    (0.2, 0.3),
    (0.3, 0.5),
    (0.5, 1),
    (1, 2),
    (2, 9),
]:
    s = j[(j.ext > lo) & (j.ext <= hi)]
    rows.append(
        dict(
            lsstcam_ext=f"{max(lo,0):g}–{min(hi,5):g}",
            exposures=len(s),
            dream_median=s.ext_dream.median(),
            dream_q10=s.ext_dream.quantile(0.1),
            dream_q90=s.ext_dream.quantile(0.9),
            P_dream_gt_0p1=(s.ext_dream > 0.1).mean(),
            P_dream_gt_0p3=(s.ext_dream > 0.3).mean(),
            P_allskycam_gt_T=(s.cam > CAM_T).mean(),
        )
    )
tw = pd.DataFrame(rows).set_index("lsstcam_ext")
print("\nDREAM field extinction and allskycam index by LSSTCam extinction bin (mag):")
print(tw.round(3).T.to_string())
for mu, lab in [(False, "moon down"), (True, "moon up  ")]:
    s = j[j.moon_up == mu]
    e3 = s.ext > 0.3
    print(
        f"{lab}: {len(s):,} exposures, P(LSSTCam ext > 0.3) = {e3.mean():.3f};  P(DREAM field > 0.3 | ext > 0.3) = {(s.ext_dream[e3] > .3).mean():.2f} vs P(allskycam > {CAM_T} | ext > 0.3) = {(s.cam[e3] > CAM_T).mean():.2f};  "
        f"P(DREAM > 0.3 | ext > 1) = {(s.ext_dream[s.ext > 1] > .3).mean():.2f} vs P(allskycam > {CAM_T} | ext > 1) = {(s.cam[s.ext > 1] > CAM_T).mean():.2f};  P(allskycam > {CAM_T} | photometric) = {(s.cam[s.ext < .1] > CAM_T).mean():.3f}, P(DREAM > 0.3 | photometric) = {(s.ext_dream[s.ext < .1] > .3).mean():.4f}"
    )
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
x = np.arange(len(fa))
axes[0].fill_between(
    x, fa.q10, fa.q90, color=P["orange"], alpha=0.15, lw=0, label="10–90 % of frames"
)
axes[0].plot(x, fa.dream_med_ext, color=P["orange"], marker="o", ms=5, label="median")
axes[0].axhline(DREAM_T, color=GRID, lw=1)
axes[0].text(
    0.05, DREAM_T + 0.1, f"DREAM cloudy threshold {DREAM_T} mag", color=INK2, fontsize=8
)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"{iv.left:g}–{iv.right:g}" for iv in fa.index], fontsize=8)
axes[0].set_xlabel("allskycam cloud index")
axes[0].set_ylabel("DREAM median extinction, alt > 30° (mag)")
axes[0].set_title(f"DREAM vs allskycam, {len(dc):,} frames", fontsize=10)
axes[0].legend(frameon=False, fontsize=8.5, loc="upper left")
x = np.arange(len(tw))
mids = np.array([0.025, 0.075, 0.15, 0.25, 0.4, 0.75, 1.5, 3.0])
axes[1].fill_between(
    x,
    tw.dream_q10,
    tw.dream_q90,
    color=P["orange"],
    alpha=0.15,
    lw=0,
    label="10–90 % of exposures",
)
axes[1].plot(
    x,
    tw.dream_median,
    color=P["orange"],
    marker="o",
    ms=5,
    label="DREAM at the pointing (median)",
)
axes[1].plot(x, mids, color=GRID, lw=1, ls="--", label="1:1")
axes[1].plot(x, 0.76 * mids, color=INK2, lw=1, ls=":", label="0.76 × LSSTCam")
axes[1].set_xticks(x)
axes[1].set_xticklabels(tw.index, fontsize=8)
axes[1].set_xlabel("LSSTCam extinction (mag, vs the band's clear-sky zero point)")
axes[1].set_ylabel("DREAM extinction at the pointing (mag)")
axes[1].set_title(f"DREAM vs LSSTCam, {len(j):,} exposures", fontsize=10)
axes[1].legend(frameon=False, fontsize=8.5, loc="upper left")
plt.tight_layout()

**Reading.** The two monitors see the same sky. Frame by frame the allskycam index and DREAM's median extinction correlate at r = 0.90, and on partial-cloud frames the cell-by-cell pattern matches at r ≈ 0.67 (the examples in `allsky/figures/` show the same gaps in both). The disagreements have the expected signs and LSSTCam adjudicates both:

* **DREAM-only cloud (68 h, 2 % of the overlap)** is thin or uniform cloud: the allskycam index is 0.17 (stars dimmed, still detected) while DREAM reports 0.8 mag, LSSTCam was observing through it half the time, and 94 % of those exposures carry > 0.3 mag of extinction (median 0.93 mag). This is the ~60 % of extincted exposures the allskycam misses in §4.1, now measured directly. Only 10 % of these minutes are moonlit.
* **Allskycam-only cloud (45 h, 58 h in v3)** is 94 % moonlit: the bright-moon completeness floor of §4.1. DREAM sees 0.12 mag, LSSTCam sees 0.33 mag median, and 54 % of the exposures do exceed 0.3 mag, so it is about half real thin cloud under the moon and half the artefact.
* On **photometric** LSSTCam exposures DREAM reports 0.04 ± 0.06 mag with no false positives above 0.3 mag (5 in 10 000), and the allskycam at index 0.3 does as well (0.1 %, 0.5 % in v3). DREAM is the better *transparency* monitor: for exposures LSSTCam measures above 0.3 mag of extinction it fires 79 % of the time with the moon down (allskycam 38 %), and above 1 mag 95 % (allskycam 65 %); with the moon up the two converge (58 vs 49 %) because DREAM's own photometric zero point degrades under moonlight. DREAM under-reads LSSTCam by a constant factor **≈ 0.75** (median ratio 0.74 over 0.3–3 mag; 0.78 moon down, 0.64 moon up): a 1.75° disc of an NSIDE-32 map smooths the cloud that the 3.5° LSSTCam field averages differently, and DREAM's reference zero point is itself set on slightly non-photometric nights.

So the allskycam measures *unobservable* sky (stars gone) and DREAM measures *non-photometric* sky (stars dimmed); at a 0.5 mag threshold they describe the same cloudy hours, and the 0.1 mag DREAM threshold reaches into its zero-point floor (10 % of photometric exposures, and a January floor of 18 % of dark time with the allskycam at 0) and should not be used for downtime.


In [ ]:
# (d) against the Scheduler WEATHER flag (2026-02-25 → 06-21, where DREAM runs), on the same minutes as the allskycam
dm1 = dm[["frac_alt30_ext_gt_0p3", "med_ext_alt30"]].resample("1min").mean()
g26["dream_ext"] = dm1.med_ext_alt30.reindex(
    g26.index, method="nearest", tolerance=pd.Timedelta("90s")
)
g26["dream_f03"] = dm1.frac_alt30_ext_gt_0p3.reindex(
    g26.index, method="nearest", tolerance=pd.Timedelta("90s")
)
bd = g26[~g26.D & ~g26.U].dropna(subset=["dream_ext", "cam"])
print(
    f"{len(bd)/60:.0f} dark hours with Scheduler status, allskycam and DREAM (DREAM covers {g26.dream_ext.notna().mean():.0%} of the flag-era minutes; it stopped on 2026-06-21): P(WEATHER) = {bd.W.mean():.3f}"
)
rows = []
for lab, pred in [
    (f"allskycam index > {CAM_T}", bd.cam > CAM_T),
    ("allskycam index > 0.5", bd.cam > 0.5),
    ("allskycam index > 0.6", bd.cam > 0.6),
    ("DREAM median ext > 0.1 mag", bd.dream_ext > 0.1),
    ("DREAM median ext > 0.3 mag", bd.dream_ext > 0.3),
    (f"DREAM median ext > {DREAM_T} mag", bd.dream_ext > DREAM_T),
    ("DREAM median ext > 1 mag", bd.dream_ext > 1.0),
    ("DREAM: > 50 % of sky above 0.3 mag", bd.dream_f03 > 0.5),
    (
        f"allskycam > {CAM_T} OR DREAM > {DREAM_T}",
        (bd.cam > CAM_T) | (bd.dream_ext > DREAM_T),
    ),
    (
        f"allskycam > {CAM_T} AND DREAM > {DREAM_T}",
        (bd.cam > CAM_T) & (bd.dream_ext > DREAM_T),
    ),
]:
    rows.append(
        dict(
            proxy=lab,
            hours=pred.sum() / 60,
            WEATHER_h=bd.W.sum() / 60,
            recall=pred[bd.W].mean(),
            precision=bd.W[pred].mean(),
            P_proxy_given_OPERATIONAL=pred[bd.status == 2].mean(),
            kappa=kappa(bd.W.values, pred.values),
        )
    )
fl = pd.DataFrame(rows).set_index("proxy")
print(fl.round(3).to_string())
cc, dd = bd.cam > CAM_T, bd.dream_ext > DREAM_T
print(
    f"\nP(WEATHER | both cloudy) = {bd.W[cc & dd].mean():.2f} ({(cc & dd).sum()/60:.0f} h);  P(WEATHER | allskycam only) = {bd.W[cc & ~dd].mean():.2f} ({(cc & ~dd).sum()/60:.0f} h);  "
    f"P(WEATHER | DREAM only) = {bd.W[~cc & dd].mean():.2f} ({(~cc & dd).sum()/60:.0f} h);  P(WEATHER | neither) = {bd.W[~cc & ~dd].mean():.3f}"
)
print(
    f"DREAM median extinction while WEATHER (inside limits) — quartiles: {bd.dream_ext[bd.W & ~bd.lim].quantile([.25,.5,.75]).round(2).tolist()};  while OPERATIONAL: {bd.dream_ext[bd.status == 2].quantile([.25,.5,.75]).round(3).tolist()}"
)

In [ ]:
# (e) nightly: fraction of dark time cloudy by each instrument, on the nights both cover ≥ 80 % of the dark minutes
dc["union"] = dc.cam_c | dc.dr_c
dc["inter"] = dc.cam_c & dc.dr_c
dc["dr_thin"] = dc.dream_med_ext_alt30 > 0.1
dn = dc.groupby("day_obs").agg(
    n=("cam_c", "size"),
    cam_frac_d=("cam_c", "mean"),
    cam_frac50_d=("cloud_idx", lambda s: (s > 0.5).mean()),
    dream_frac=("dr_c", "mean"),
    dream_frac_thin=("dr_thin", "mean"),
    union_frac=("union", "mean"),
    inter_frac=("inter", "mean"),
)
dn["dark_h"] = alm.dark_hours_12.reindex(dn.index)
dn["cov"] = dn.n / 60 / dn.dark_h
dv = dn[dn["cov"] > 0.8].copy()
dv["month"] = dv.index.astype(str).str[4:6].astype(int)
eqm = (
    dv.groupby("month")[
        [
            "cam_frac_d",
            "cam_frac50_d",
            "dream_frac",
            "dream_frac_thin",
            "union_frac",
            "inter_frac",
        ]
    ]
    .mean()
    .reindex(range(1, 13))
)
eq = eqm.mean()
print(
    f"{len(dv)} nights ({dv.index.min()} → {dv.index.max()}) where DREAM and the allskycam both cover ≥ 80 % of the dark minutes. Fraction of dark time cloudy, equal-month mean:\n"
    f"   allskycam > {CAM_T}: {eq.cam_frac_d:.3f} | allskycam > 0.5: {eq.cam_frac50_d:.3f} | DREAM > {DREAM_T} mag: {eq.dream_frac:.3f} | DREAM > 0.1 mag (thin, at DREAM's zero-point floor): {eq.dream_frac_thin:.3f} | either: {eq.union_frac:.3f} | both: {eq.inter_frac:.3f}"
)
print(
    f"night by night r(allskycam, DREAM) = {np.corrcoef(dv.cam_frac_d, dv.dream_frac)[0,1]:.2f}; nights fully lost (> 0.9): allskycam {(dv.cam_frac_d > .9).mean():.3f}, DREAM {(dv.dream_frac > .9).mean():.3f}, either {(dv.union_frac > .9).mean():.3f}; "
    f"clear (< 0.1): {(dv.cam_frac_d < .1).mean():.2f} / {(dv.dream_frac < .1).mean():.2f} / {(dv.union_frac < .1).mean():.2f}"
)
big = dv[(dv.cam_frac_d - dv.dream_frac).abs() > 0.3]
print(
    f"nights where the two differ by > 0.3 of the night: {len(big)} — DREAM higher: {list(big.index[big.dream_frac > big.cam_frac_d])}; allskycam higher: {list(big.index[big.cam_frac_d > big.dream_frac])}"
)
print(eqm.round(3).T.to_string())
nights = nights.join(dv[["dream_frac", "union_frac", "inter_frac"]])
dream_res = dict(
    nights=int(len(dv)),
    span=[int(dv.index.min()), int(dv.index.max())],
    dream_threshold_mag=DREAM_T,
    equal_month_mean=eq.round(4).to_dict(),
    monthly=eqm.round(4).to_dict(),
    flag_calibration=fl.round(4).to_dict(orient="index"),
    lsstcam_bins=tw.round(4).to_dict(orient="index"),
    classes=dis.round(4).to_dict(orient="index"),
)
dv.to_parquet(DATA / "dream_nightly.parquet")
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
mo = np.arange(1, 13)
axes[0].plot(
    mo,
    eqm.cam_frac_d * 100,
    color=P["blue"],
    marker="o",
    ms=4,
    label=f"allskycam index > {CAM_T}",
)
axes[0].plot(
    mo,
    eqm.dream_frac * 100,
    color=P["orange"],
    marker="o",
    ms=4,
    label=f"DREAM median ext > {DREAM_T} mag",
)
axes[0].plot(mo, eqm.union_frac * 100, color=INK2, lw=1.2, ls="--", label="either")
axes[0].plot(
    mo,
    eqm.dream_frac_thin * 100,
    color=P["orange"],
    lw=1.2,
    ls=":",
    label="DREAM > 0.1 mag (thin cloud)",
)
axes[0].set_xticks(mo)
axes[0].set_xticklabels(list("JFMAMJJASOND"))
axes[0].set_ylabel("% of dark time cloudy")
axes[0].set_title(
    f"Monthly, {len(dv)} nights with both instruments (Jun 2025 → Jun 2026)",
    fontsize=10,
)
axes[0].legend(frameon=False, fontsize=8.5)
axes[1].plot([0, 1], [0, 1], color=GRID, lw=1)
axes[1].scatter(
    dv.cam_frac_d,
    dv.dream_frac,
    s=22,
    color=P["blue"],
    edgecolor=SURF,
    linewidth=0.8,
    zorder=3,
)
for d, r in big.iterrows():
    axes[1].annotate(
        str(d),
        (r.cam_frac_d, r.dream_frac),
        textcoords="offset points",
        xytext=(4, -3),
        fontsize=6.5,
        color=INK2,
    )
axes[1].set_xlabel(f"allskycam: fraction of dark time with index > {CAM_T}")
axes[1].set_ylabel(f"DREAM: fraction with median ext > {DREAM_T} mag")
axes[1].set_title(
    f"Night by night, r = {np.corrcoef(dv.cam_frac_d, dv.dream_frac)[0,1]:.2f}",
    fontsize=10,
)
plt.tight_layout()

**Reading.** Against the Scheduler flag DREAM behaves like a less specific allskycam: at any threshold its recall is comparable (0.52–0.70) but its precision is lower (0.31–0.43 vs 0.48–0.57) and κ peaks at 0.40 (1 mag) against 0.52 for the allskycam at 0.5. The reason is in the two-way split: hours both instruments call cloudy are flagged 46 % of the time, allskycam-only hours 56 %, but DREAM-only hours just 7 %, the thin cloud through which the survey keeps observing (above). The operators close for *lost stars*, not for extinction, which is also what §4.2 found from the κ peak at index 0.5. The union of the two does not help (κ 0.45) because it adds exactly the thin-cloud hours the operators ignore; the intersection is the most specific proxy of all (precision 0.46) at the cost of recall.

At the night level the two agree at r = 0.91 over 283 nights, with the same bimodality (80 % of nights clear by either, 2–3 % fully lost). Over Jun 2025 → Jun 2026 DREAM at 0.5 mag gives **0.106** of dark time cloudy against the allskycam's 0.098 (0.075 at index 0.5); either instrument 0.120, both 0.083. The eight nights that differ by more than 0.3 split cleanly: DREAM higher on five thin-cloud nights (2025-09-12, 2025-11-17, 2026-05-19, -05-22, -06-12: where LSSTCam observed, 54–100 % of exposures had > 0.3 mag of extinction while the allskycam index stayed low) and the allskycam higher on three full-moon nights (2026-03-04, -05-27, -05-29; moon altitude up to 64–84°, DREAM 0.0–0.1 mag), the moon floor again. **Conclusion for §5:** adding DREAM does not change the headline. Its one year overlaps the clear 2025–26 season, it adds ≈ +0.02 of dark time as thin cloud when taken as a union, and it confirms the allskycam's bright-moon floor as the dominant known bias. DREAM's own value is as the *transparency* record: the fraction of observed exposures that are non-photometric, which the §5 "unobservable time" numbers deliberately exclude. `data/dream_nightly.parquet` carries the per-night fractions and the result JSON a `dream` block.


## 5. Long-baseline weather downtime for the scheduler simulations

Four baselines, each giving a **nightly fraction of dark time lost to weather**:

| baseline | span | what it measures |
|---|---|---|
| **Scheduler WEATHER flag** | 2026-02-25 → 07-13 (4.5 months, austral autumn/winter only) | the operational truth, but seasonally biased |
| **Allskycam** (cloud index > 0.3, and > 0.5) | 2021-07 → 2026-07 (1523 nights with ≥ 80 % frame coverage that pass the astrometric gate) | the measured sky, validated against the flag (§4.2) and LSSTCam photometry (§4.1); the humidity / dew-point / wind limits are added where the EFD overlaps (1134 nights) |
| **EFD proxy** (limits ∪ cloud nowcast > 70 %) | 2023-03 → 2026-07 (827 nights with full coverage; limits-only back to 2019-12, 1581 nights) | the previous estimate, kept for comparison |
| **CTIO cloud database** (`clouds_ctio_1975_2022.db`, cloud > 0.3 = the `ModelObservatory` default `cloud_limit`) | 1975 → 2022, 17 414 nights | what the scheduler simulations assume today |

Note that `rubin_scheduler.site_models.UnscheduledDowntimeData` models *hardware* downtime only; weather enters the simulations exclusively through the cloud database, with no wind/humidity model (the default `wind_data` is "no wind").

In [ ]:
pv = proxy[proxy.valid_full].copy()
pv["month"] = pv.index.astype(str).str[4:6].astype(int)
pv["year"] = pv.index.astype(str).str[:4].astype(int)
pl = proxy[proxy.valid_lim].copy()
pl["month"] = pl.index.astype(str).str[4:6].astype(int)
pl["year"] = pl.index.astype(str).str[:4].astype(int)
cl = pd.read_parquet(DATA / "ctio_clouds_1975_2022.parquet")
cl["bad"] = cl.cloud > 0.3
cl["night"] = (cl.t - pd.Timedelta(hours=16)).dt.date  # local night key
cn = cl.groupby("night").agg(
    bad=("bad", "mean"),
    month=("t", lambda s: s.iloc[0].month),
    year=("t", lambda s: s.iloc[0].year),
)
sf = nights.loc[FLAG_START:DAY1].dropna(subset=["W_h"])
sf = sf[sf.D_h < 0.5 * sf.night_h]
sf_frac = sf.W_h / (sf.night_h - sf.D_h)
clim = pd.DataFrame(
    {
        "Scheduler WEATHER 2026": sf_frac.groupby(
            sf.index.astype(str).str[4:6].astype(int)
        ).mean(),
        "EFD proxy 2023–26 (limits ∪ cloud>70)": pv.groupby("month").bad_frac.mean(),
        "EFD limits only 2019–26 (RH/Td/wind)": pl.groupby("month").lim_frac.mean(),
        "CTIO clouds 1975–2022 (cloud>0.3)": cn.groupby("month").bad.mean(),
        f"Allskycam 2021–26 (index>{CAM_T})": camn[camn.cam_valid]
        .groupby("month")
        .cam_frac.mean(),
        "Allskycam 2021–26 (index>0.5)": camn[camn.cam_valid]
        .groupby("month")
        .cam_frac50.mean(),
        f"Allskycam ∪ limits 2021–26 (index>{CAM_T})": proxy[proxy.valid_cam_full]
        .assign(month=lambda d: d.index.astype(str).str[4:6].astype(int))
        .groupby("month")
        .bad_cam_frac.mean(),
    }
).reindex(range(1, 13))
clim.index.name = "month"
pc = proxy[proxy.valid_cam_full].copy()
pc["month"] = pc.index.astype(str).str[4:6].astype(int)
pc["year"] = pc.index.astype(str).str[:4].astype(int)
cv = camn[camn.cam_valid]
clim.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
mo = np.arange(1, 13)
C = clim.columns
ax.fill_between(
    mo,
    clim[C[5]] * 100,
    clim[C[4]] * 100,
    color=P["blue"],
    alpha=0.13,
    lw=0,
    label="allskycam: index > 0.5 … > 0.3",
)
ax.plot(
    mo,
    clim[C[4]] * 100,
    color=P["blue"],
    lw=2.4,
    marker="o",
    ms=5,
    label=f"allskycam 2021–26, dark time with cloud index > {CAM_T}",
)
ax.plot(
    mo,
    clim[C[6]] * 100,
    color=P["blue"],
    lw=1,
    ls=":",
    label="allskycam ∪ humidity / dew-point / wind limits",
)
ax.plot(
    mo,
    clim[C[3]] * 100,
    color=INK2,
    lw=2,
    label="CTIO clouds 1975–2022 (cloud > 0.3, scheduler default)",
)
ax.plot(
    mo,
    clim[C[1]] * 100,
    color=P["aqua"],
    lw=1.6,
    ls="--",
    label="meteoblue nowcast proxy 2023–26 (limits ∪ cloud > 70 %)",
)
ax.plot(
    mo,
    clim[C[2]] * 100,
    color=P["violet"],
    lw=1.2,
    label="EFD limits only 2019–26 (humidity / dew point / wind)",
)
ax.plot(
    mo,
    clim[C[0]] * 100,
    color=P["orange"],
    lw=0,
    marker="s",
    ms=7,
    label="Scheduler WEATHER flag, Mar–Jul 2026",
)
ax.set_xticks(mo)
ax.set_xticklabels(
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
)
ax.set_ylabel("% of dark time lost to weather")
ax.set_title("Seasonal weather-downtime climatology at Cerro Pachón")
ax.legend(frameon=False, fontsize=8.3, loc="upper left")
ax.set_ylim(0, 60)
plt.tight_layout()

### 5.1 Headline statistics for the simulations

Nightly fraction of dark time lost to weather (per-night distribution and the annual mean). Every series is weighted so each calendar month counts equally (allskycam and proxy coverage is uneven); the nightly standard deviation is the raw per-night spread. The year-to-year spread is estimated the same way for every series, as the standard deviation over years of the mean monthly anomaly against the series' own monthly climatology (months with ≥ 10 nights, years with ≥ 6 such months), so that partial years can be used; for CTIO this reproduces the plain standard deviation of its 48 complete years.

In [ ]:
def stats(frac_by_night, months):
    m = pd.Series(months, index=frac_by_night.index)
    eq = frac_by_night.groupby(m).mean().reindex(range(1, 13))
    return dict(
        nights=len(frac_by_night),
        mean_equal_month=eq.mean(),
        mean_raw=frac_by_night.mean(),
        median=frac_by_night.median(),
        std_nightly=frac_by_night.std(),
        frac_nights_fully_lost=(frac_by_night > 0.9).mean(),
        frac_nights_partial=((frac_by_night > 0.1) & (frac_by_night <= 0.9)).mean(),
        frac_nights_clear=(frac_by_night <= 0.1).mean(),
    )


def annual_spread(frac, years, months, min_nights=10, min_months=6):
    """Year-to-year std of the annual fraction, from each year's mean monthly anomaly vs the series' own monthly climatology (months with ≥ min_nights nights; years with ≥ min_months such months). Robust to uneven coverage."""
    d = pd.DataFrame(
        dict(f=np.asarray(frac), y=np.asarray(years), m=np.asarray(months))
    )
    mm = d.groupby(["y", "m"]).f.agg(["mean", "size"])
    mm = mm[mm["size"] >= min_nights]
    clim_m = mm["mean"].groupby(level=1).mean()
    an = (
        (mm["mean"] - clim_m.reindex(mm.index.get_level_values(1)).values)
        .groupby(level=0)
        .mean()
    )
    an = an[mm.groupby(level=0).size() >= min_months]
    return float(an.std()), int(len(an))


yr = cn.groupby("year").bad.mean()
cam_yr = cv.groupby("year").agg(
    nights=("cam_frac", "size"),
    frac=("cam_frac", "mean"),
    frac50=("cam_frac50", "mean"),
)
head = pd.DataFrame(
    {
        "Scheduler WEATHER flag (Mar–Jul 2026 only)": stats(
            sf_frac, sf.index.astype(str).str[4:6].astype(int)
        ),
        f"Allskycam 2021–26 (index>{CAM_T})": stats(cv.cam_frac, cv.month),
        "Allskycam 2021–26 (index>0.5)": stats(cv.cam_frac50, cv.month),
        f"Allskycam ∪ limits 2021–26 (index>{CAM_T})": stats(pc.bad_cam_frac, pc.month),
        f"Allskycam, LSSTCam window 2025-04→2026-07 (index>{CAM_T})": stats(
            cv.loc[DAY0:DAY1].cam_frac, cv.loc[DAY0:DAY1].month
        ),
        "Nowcast proxy 2023–26 (limits ∪ cloud>70)": stats(pv.bad_frac, pv.month),
        "Nowcast proxy, LSSTCam window": stats(
            pv.loc[DAY0:DAY1].bad_frac, pv.loc[DAY0:DAY1].month
        ),
        "CTIO clouds 1975–2022 (sim default)": stats(cn.bad, cn.month),
    }
).T
sp = [
    (np.nan, 0),
    annual_spread(cv.cam_frac, cv.year, cv.month),
    annual_spread(cv.cam_frac50, cv.year, cv.month),
    annual_spread(pc.bad_cam_frac, pc.year, pc.month),
    (np.nan, 0),
    annual_spread(pv.bad_frac, pv.year, pv.month),
    (np.nan, 0),
    annual_spread(cn.bad, cn.year, cn.month),
]
head["annual_std_across_years"] = [s[0] for s in sp]
head["years_in_annual_std"] = [s[1] for s in sp]
print(
    f"CTIO: plain std of the 48 complete-year means = {yr.std():.3f} (the month-anomaly estimator gives {sp[-1][0]:.3f})"
)
print("allskycam by year:")
print(cam_yr.round(3).T.to_string())
head.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
bins = np.linspace(0, 1, 21)
axes[0].hist(
    cv.cam_frac,
    bins=bins,
    color=P["blue"],
    zorder=2,
    density=True,
    label=f"allskycam 2021–26 (index > {CAM_T})",
)
axes[0].hist(
    cn.bad,
    bins=bins,
    histtype="step",
    color=INK2,
    lw=2,
    zorder=3,
    density=True,
    label="CTIO 1975–2022",
)
axes[0].hist(
    pv.bad_frac,
    bins=bins,
    histtype="step",
    color=P["aqua"],
    lw=1.6,
    ls="--",
    zorder=3,
    density=True,
    label="nowcast proxy 2023–26",
)
axes[0].set_xlabel("fraction of night lost to weather")
axes[0].set_ylabel("density")
axes[0].legend(frameon=False, fontsize=8.5)
axes[0].set_title("Per-night distribution is bimodal: clear or lost")
axes[1].plot(
    yr.index, yr.values * 100, color=INK2, lw=1.5, marker="o", ms=3, label="CTIO years"
)
axes[1].scatter(
    cam_yr.index[cam_yr.nights >= 300],
    cam_yr.frac[cam_yr.nights >= 300] * 100,
    color=P["blue"],
    s=46,
    zorder=5,
    edgecolor=SURF,
    label="allskycam (complete years)",
)
axes[1].scatter(
    cam_yr.index[cam_yr.nights < 300],
    cam_yr.frac[cam_yr.nights < 300] * 100,
    color=P["blue"],
    s=46,
    zorder=5,
    facecolor=SURF,
    linewidth=1.5,
    label="allskycam (partial years)",
)
pyr = pv.groupby("year").bad_frac.mean()
axes[1].scatter(
    pyr.index,
    pyr.values * 100,
    color=P["aqua"],
    s=22,
    zorder=4,
    marker="D",
    label="nowcast proxy",
)
axes[1].axhline(yr.mean() * 100, color=GRID, lw=1)
axes[1].set_ylabel("% dark time lost")
axes[1].set_title("Annual mean fraction of dark time lost")
axes[1].legend(frameon=False, fontsize=8, loc="upper left")
plt.tight_layout()

### 5.2 Recommendation for scheduler-simulation inputs

> **Revised 2026-10-04 with the allskycam, and 2026-10-11 with the v5 re-reduction.** The earliest version of this section rested on the meteoblue nowcast proxy and recommended 0.155; the allskycam shows the nowcast misses thin and broken cloud (§4.2, §4.4). The 2026-10-04 version used the v3 reduction and recommended 0.17 / 0.13; v5 removed false cloud from camera-swap months and from soft-focus epochs, and the numbers below are for v5.

* **Point estimate:** the allskycam gives **≈ 0.14 of dark time with unusable sky** (equal-month annual mean 0.143 at cloud index > 0.3; 0.16 with the humidity / dew-point / wind limits added), with **≈ 6 % of nights fully lost, ≈ 18 % partially affected, ≈ 76 % clear**. At the stricter index > 0.5, which matches the operators' flagged hours to within 7 % and is where κ against the flag peaks (§4.2), it gives **0.12**. Over the LSSTCam window itself (Apr 2025 → Jul 2026) the allskycam gives 0.13 (equal-month; 0.16 of dark hours by direct accounting), and the whole-night accounting of §3 gives ≈ 18 % of all dark hours (57 fully-lost weather nights ≈ 13.3 %, partial losses on observed nights ≈ 5 %; whole-night accounting rounds every > 50 % night up to a full loss and so leans high). **Recommended central value: 0.14 at the allskycam's "stars not detectable" threshold, 0.12 at the operators' de-facto closing threshold, with a ± 0.04 year-to-year spread.** Use the allskycam monthly table (`data/weather_downtime_monthly_climatology.csv`, both thresholds), not a single number.
* **Seasonality dominates:** Jun ≈ 35 %, Aug ≈ 30 %, Jul ≈ 28 %, May ≈ 22 %, Sep ≈ 16 %, Oct ≈ 12 %, Jan–Mar ≈ 2–3 %. The secondary February peak of the v3 record (≈ 11 %) was the February 2023 camera-swap artefact and is gone. The *per-night* standard deviation (≈ 0.28) reflects the clear-or-lost bimodality rather than a Gaussian spread.
* **Year-to-year:** the six allskycam years (2021–26) span 0.10–0.18 (2021 low, 2023 high) with a month-anomaly standard deviation of ≈ 0.04; the CTIO record's 48 years give 0.05 around a mean of 0.25. A ± 0.04–0.05 annual spread is the appropriate input; six years do not resolve whether the Rubin era is intrinsically less variable or merely short.
* **The scheduler's default CTIO cloud model is ≈ 11 percentage points more pessimistic** than the Rubin record (0.25 vs 0.14 equal-month means; CTIO fully-lost nights 16 % vs 6 %) and its seasonal shape differs: it has 2.5–5× too much cloud in Jan–Apr and 2–2.4× in Sep–Nov (0.08–0.34 vs the allskycam's 0.02–0.16) and is closest to the record only in Jun and Aug (1.2×). An eye-estimated 3/8-okta threshold fires on partial cloud through which Rubin observes. A scaled CTIO model does *not* reproduce the Rubin shape; the allskycam monthly table should be used directly.
* **Humidity / dew point / wind alone account for ≈ 6 %** of dark time (equal-month mean of the limits-only column) and mostly coincide with cloud: adding them to the allskycam raises the annual fraction by only 0.02. A wind/humidity model in `ModelObservatory` would add little compared with getting the cloud fraction right.

**Caveats.** The cloud index is a star-count metric: thin uniform cirrus that dims stars without hiding them registers weakly (the allskycam flags only ~36 % of LSSTCam exposures with > 0.3 mag extinction, §4.1), so 0.14 is a lower bound on non-photometric time and an estimate of *unobservable* time. DREAM (§4.5) measures that thin cloud directly: over its one-year overlap it adds ≈ 0.02 of dark time as thin, observed-through cloud when taken as a union with the allskycam, and the operators do not close for it (7 % of DREAM-only hours are flagged), so the 0.14 / 0.12 recommendation stands. Under a bright moon the completeness model still leaves a residual index floor on some frames: on skies brighter than 12 ADU/s (6 % of dark frames) a photometric LSSTCam exposure has a 6 % chance of an allskycam index above 0.3, against 0.1 % otherwise, so the bias on the annual fraction is ≈ +0.004, and at most +0.03 if every index > 0.3 frame under such a sky were false; DREAM confirms the floor (the allskycam-only cloudy hours are 94 % moonlit, §4.5). The 70 % nowcast threshold and the flag comparison are tuned to one season of operator behaviour (Mar–Jul 2026). The analysis excludes the July 2026 storm and everything after, as intended.

In [ ]:
out = {
    "window": [DAY0, DAY1],
    "limits": LIM,
    "cloud_threshold_pct": CLOUD_T,
    "allskycam_index_threshold": CAM_T,
    "monthly_fraction_lost": clim.round(4).to_dict(),
    "headline": head.round(4).to_dict(orient="index"),
    "nights_class_counts": nights["class"].value_counts().to_dict(),
    "nights_class_counts_without_allskycam": nights.class_nowcast.value_counts().to_dict(),
    "flag_calibration": {
        "nowcast": scan.to_dict(orient="index"),
        "allskycam": scan_cam.to_dict(orient="index"),
    },
    "envelope": envelope.astype(str).to_dict(orient="index"),
    "dream": dream_res,
}
json.dump(out, open(DATA / "weather_downtime_results.json", "w"), indent=1, default=str)
nights.drop(columns=["t_set", "t_rise"]).to_parquet(
    DATA / "nightly_attribution_20250415_20260713.parquet"
)
clim.to_csv(DATA / "weather_downtime_monthly_climatology.csv")
print(
    "saved data/weather_downtime_results.json, data/weather_downtime_monthly_climatology.csv, data/nightly_attribution_20250415_20260713.parquet"
)